# Phase-4 — Execution-Verified Generation + Calibrated Abstention (EVG-CAS)

**Paper:** *Trustworthy Small Language Models for Financial Tasks*
**Phase:** 4 — proposed method (guide §15 stages 12 & 18)
**Backbone (frozen by Phase-3):** **E5** — `Qwen/Qwen3-1.7B` + balanced joint QLoRA
**Prompt policy (frozen):** FinQA `canonical` · FiQA `explicit_schema`

Phase-3 established *that* joint QLoRA works and *where* it fails. Phase-4 proposes an
inference-time method that fixes the two failures Phase-3 measured, without touching the
frozen backbone, adapter, prompt policy or selection rule.

| Component | Idea | Phase-3 defect it targets |
|---|---|---|
| **EVG** — execution-verified generation | sample *n* candidate programs, execute each on the **input table** with the official FinQA executor, emit the majority answer | E12: 96 % of errors are execution/reasoning, 0.3 % invalid programs |
| **CAS** — calibrated abstention | Platt / isotonic calibrated on **validation only** with 5-fold cross-fitting → risk-coverage + AURC | E14: confidence ROC-AUC only 0.577 |
| **EVG+CAS** | vote agreement becomes a calibrator feature | C3: verification and abstention are complementary |

Full design, pre-declared novelty claims and guardrails: **`../trust_fin_extra_phase/PLAN.md`**
(in this folder: `PLAN.md`).

---

## How to run

```bash
cd "/home/heet18/Downloads/Research Papers/trust_fin_extra_phase"

# 1. validation-only stages — order matters: sample first, then calibrate, then freeze
TRUSTFIN4_RUN_MODE=full  /home/heet18/trustfin-slm/.venv/bin/python tools/phase4_shard.py run P4EVG   # E2, E3, E3b  (GPU ~1-1.5 h)
TRUSTFIN4_RUN_MODE=full  /home/heet18/trustfin-slm/.venv/bin/python tools/phase4_shard.py run P4CAL   # E1, E4-E7    (CPU, minutes)
# smoke: same two commands with TRUSTFIN4_RUN_MODE=smoke

# 2. ONLY after phase4_freeze.json says FROZEN (two-key opt-in):
TRUSTFIN4_PROTECTED=1 TRUSTFIN4_RUN_MODE=full \
  /home/heet18/trustfin-slm/.venv/bin/python tools/phase4_shard.py run P4TEST  # E8 (GPU ~2-3 h)
```

**Read contract.** Everything under `../artifacts/` and `../data/` is opened **read-only**.
All Phase-4 output is written under `trust_fin_extra_phase/artifacts/phase4/`.

**Guardrails (asserted, not aspirational).** Protected test keys cannot be resolved unless
`phase4_freeze.json.status == "FROZEN"` **and** `TRUSTFIN4_PROTECTED=1`. No hyper-parameter
below was chosen after seeing any protected record — `n`, temperature, calibrator family and
coverage targets are pre-declared in `PLAN.md` §4/§6.

## 0. Dependency check

In [1]:
# Set True only if the environment lacks these packages.
INSTALL_DEPS = False
if INSTALL_DEPS:
    %pip install -q transformers peft bitsandbytes accelerate datasets scikit-learn matplotlib

## 1. Imports, deterministic seed, environment capture

In [2]:
from __future__ import annotations

import os, gc, re, json, math, time, random, hashlib, platform, importlib.util, shutil
from pathlib import Path
from collections import Counter, defaultdict
from typing import Any, Dict, List, Mapping, Optional, Sequence, Tuple

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

from scipy.stats import pearsonr, spearmanr
from sklearn.metrics import (
    mean_absolute_error, mean_squared_error, r2_score,
    accuracy_score, f1_score, roc_auc_score, brier_score_loss,
)
from sklearn.model_selection import KFold
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.isotonic import IsotonicRegression

from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel

SEED = 42

def seed_everything(seed: int = SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

seed_everything()
np.set_printoptions(suppress=True)

print("Python:", platform.python_version())
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("BF16 supported:", torch.cuda.is_bf16_supported())

Python: 3.12.3
PyTorch: 2.11.0+cu130
CUDA available: True
GPU: NVIDIA GeForce RTX 4050 Laptop GPU
BF16 supported: True


## 2. Global configuration

In [3]:
# ---------------- USER CONFIGURATION ----------------
RUN_MODE = os.environ.get("TRUSTFIN4_RUN_MODE", "full").lower()   # "smoke" | "full"
OVERWRITE = os.environ.get("TRUSTFIN4_OVERWRITE", "0").strip() in {"1", "true", "True"}
# Second key required before any protected test artifact can be resolved.
PROTECTED_OPT_IN = os.environ.get("TRUSTFIN4_PROTECTED", "").strip() in {"1", "true", "True"}

PRIMARY_MODEL = "Qwen/Qwen3-1.7B"          # frozen Phase-3 backbone base model

# --- layout: this folder is the Phase-4 workspace, parent holds Phase-2/2.1/3 ---
def _find_workspace(start: Path) -> Path:
    """Anchor the workspace explicitly — nbconvert runs the kernel with cwd = the
    notebook's directory, so Path.cwd() alone would resolve to .../notebooks."""
    env = os.environ.get("TRUSTFIN4_WORKSPACE", "").strip()
    if env:
        return Path(env).expanduser().resolve()
    cur = start.resolve()
    for p in [cur, *cur.parents]:
        if (p / "PLAN.md").exists() and (p / "tools" / "phase4_shard.py").exists():
            return p
    fallback = Path("/home/heet18/Downloads/Research Papers/trust_fin_extra_phase")
    return fallback if fallback.exists() else cur

HERE = _find_workspace(Path.cwd())                           # .../trust_fin_extra_phase
OUTER = HERE.parent                                          # .../Research Papers
if not (OUTER / "artifacts/phase2_1").exists():
    OUTER = Path("/home/heet18/Downloads/Research Papers")   # explicit fallback
assert (OUTER / "artifacts/phase2_1").exists(), f"Outer workspace not found: {OUTER}"
assert (HERE / "tools" / "phase4_shard.py").exists(), f"Phase-4 workspace invalid: {HERE}"
# Smoke and full runs must never share artifact paths: a smoke freeze/candidate cache
# would otherwise silently satisfy the full run's gates.
PHASE4_ROOT = HERE / "artifacts" / ("phase4_smoke" if RUN_MODE == "smoke" else "phase4")

# --- pre-declared EVG hyper-parameters (PLAN.md §4/§6; never tuned on test) ---
EVG_N_FULL = 8
EVG_N_SMOKE = 2
EVG_TEMPERATURE = 0.7
EVG_TOP_P = 0.9
CALIB_FOLDS = 5
COVERAGE_TARGETS = [0.60, 0.70, 0.80, 0.90, 0.95]
FROZEN_COVERAGES = [0.80, 0.90]          # operating points written into the freeze
FINQA_MAX_NEW_TOKENS = 192
FIQA_MAX_NEW_TOKENS = 24
OTHER_MAX_NEW_TOKENS = 16                # FOMC stance: one label token

MAX_SEQ_LENGTH_FULL = 1024
MAX_SEQ_LENGTH_SMOKE = 768
MAX_SEQ_LENGTH = MAX_SEQ_LENGTH_SMOKE if RUN_MODE == "smoke" else MAX_SEQ_LENGTH_FULL

VAL_CAP_PER_DATASET = 20 if RUN_MODE == "smoke" else None
EVG_N = EVG_N_SMOKE if RUN_MODE == "smoke" else EVG_N_FULL

USE_4BIT = True
BNB_QUANT_TYPE = "nf4"
BNB_DOUBLE_QUANT = True

_ALL_STAGES = {"P4-E1", "P4-E2", "P4-E3", "P4-E3b", "P4-E4", "P4-E5", "P4-E6",
               "P4-E7", "P4-E8", "P4-E9"}
_stages_env = os.environ.get("TRUSTFIN4_EXPERIMENTS", "").strip()
_stage_lookup = {s.upper(): s for s in _ALL_STAGES}   # 'P4-E3B' -> 'P4-E3b'
RUN_STAGES = set()
if _stages_env:
    for tok in [x.strip() for x in _stages_env.split(",") if x.strip()]:
        key = tok.upper()
        assert key in _stage_lookup, f"Unknown stage ID: {tok!r}"
        RUN_STAGES.add(_stage_lookup[key])
else:
    RUN_STAGES = set(_ALL_STAGES)
# P4-E9 (FinTrust official eval) is explicitly deferred by PLAN.md §4 unless requested.
if "P4-E9" not in _stages_env:
    RUN_STAGES.discard("P4-E9")

assert RUN_MODE in {"smoke", "full"}
print("RUN_MODE:", RUN_MODE)
print("RUN_STAGES:", sorted(RUN_STAGES))
print("HERE (Phase-4 workspace):", HERE)
print("OUTER (read-only):", OUTER)
print("EVG_N:", EVG_N, "| temperature:", EVG_TEMPERATURE, "| top_p:", EVG_TOP_P)
print("PROTECTED_OPT_IN:", PROTECTED_OPT_IN)

RUN_MODE: full
RUN_STAGES: ['P4-E1', 'P4-E4', 'P4-E5', 'P4-E6', 'P4-E7']
HERE (Phase-4 workspace): /home/heet18/Downloads/Research Papers/trust_fin_extra_phase
OUTER (read-only): /home/heet18/Downloads/Research Papers
EVG_N: 8 | temperature: 0.7 | top_p: 0.9
PROTECTED_OPT_IN: False


## 3. Locate the frozen Phase-2.1 contract and the Phase-3 freeze

In [4]:
def read_json(path: Path):
    with Path(path).open("r", encoding="utf-8") as f:
        return json.load(f)

def write_json(path: Path, obj: Any):
    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, ensure_ascii=False, default=str)

def read_jsonl(path: Path):
    rows = []
    with Path(path).open("r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                rows.append(json.loads(line))
    return rows

def write_jsonl(path: Path, rows: Sequence[Mapping[str, Any]]):
    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as f:
        for r in rows:
            f.write(json.dumps(dict(r), ensure_ascii=False, default=str) + "\n")

def sha256_file(path: Path) -> str:
    h = hashlib.sha256()
    with Path(path).open("rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

def load_module_from_path(name: str, path: Path):
    spec = importlib.util.spec_from_file_location(name, str(path))
    if spec is None or spec.loader is None:
        raise ImportError(f"Cannot import {name} from {path}")
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    return module

# ---- Phase-2.1 data config + registry (outer, read-only) ----
P21 = OUTER / "artifacts/phase2_1/manifests"
PHASE3_DATA_CONFIG = read_json(P21 / "phase3_data_config.json")
PHASE4_DATA_CONFIG = read_json(P21 / "phase4_eval_config.json")
REGISTRY_PATH = Path(PHASE4_DATA_CONFIG["data_registry"])
REGISTRY = read_json(REGISTRY_PATH)
assert sha256_file(REGISTRY_PATH) == "e0c43a54add7323b60d139cac7be6a93694fad96d979b94539cb01968595e794", \
    "Phase-2.1 registry hash drifted"

# ---- Phase-3 freeze (the contract Phase-4 inherits) ----
P3_ROOT = OUTER / "artifacts/phase3"
PHASE3_FREEZE = read_json(P3_ROOT / "manifests/phase3_freeze.json")
assert PHASE3_FREEZE["status"] == "FROZEN", "Phase-3 is not frozen; Phase-4 must not start."
assert PHASE3_FREEZE["winner"]["experiment"] == "E5", "Unexpected Phase-3 winner."
BACKBONE_ADAPTER = Path(PHASE3_FREEZE["winner"]["adapter_path"])
assert BACKBONE_ADAPTER.exists(), f"Frozen adapter missing: {BACKBONE_ADAPTER}"
FINQA_PROMPT = PHASE3_FREEZE["prompt_policy"]["FinQA"]        # 'canonical'
FIQA_PROMPT  = PHASE3_FREEZE["prompt_policy"]["FiQA"]         # 'explicit_schema'
BASE_MODEL_ID = PHASE3_FREEZE["winner"]["model_id"]
assert BASE_MODEL_ID == PRIMARY_MODEL

# ---- output dirs (all inside this folder) ----
PRED_DIR = PHASE4_ROOT / "predictions"
METRIC_DIR = PHASE4_ROOT / "metrics"
ANALYSIS_DIR = PHASE4_ROOT / "analysis"
MANIFEST_DIR = PHASE4_ROOT / "manifests"
PLOT_DIR = PHASE4_ROOT / "plots"
for p in [PRED_DIR, METRIC_DIR, ANALYSIS_DIR, MANIFEST_DIR, PLOT_DIR]:
    p.mkdir(parents=True, exist_ok=True)

PHASE4_FREEZE_PATH = MANIFEST_DIR / "phase4_freeze.json"

print("Phase-2.1 config :", P21 / "phase3_data_config.json")
print("Phase-4 eval cfg :", P21 / "phase4_eval_config.json")
print("Phase-3 freeze   :", PHASE3_FREEZE["status"], "| winner:",
      PHASE3_FREEZE["winner"]["experiment"], "| adapter:", BACKBONE_ADAPTER)
print("Prompt policy    : FinQA=%s  FiQA=%s" % (FINQA_PROMPT, FIQA_PROMPT))
print("Phase-4 outputs  :", PHASE4_ROOT)

assert set(PHASE3_DATA_CONFIG["allowed_training_datasets"]) == {"FinQA", "FiQA"}
assert {"FinBen-FOMC", "FinTrust"}.issubset(set(PHASE3_DATA_CONFIG["forbidden_training_datasets"]))
print("PASS: Phase-2.1 training policy preserved; Phase-4 trains nothing.")

Phase-2.1 config : /home/heet18/Downloads/Research Papers/artifacts/phase2_1/manifests/phase3_data_config.json
Phase-4 eval cfg : /home/heet18/Downloads/Research Papers/artifacts/phase2_1/manifests/phase4_eval_config.json
Phase-3 freeze   : FROZEN | winner: E5 | adapter: /home/heet18/Downloads/Research Papers/artifacts/phase3/checkpoints/E5
Prompt policy    : FinQA=canonical  FiQA=explicit_schema
Phase-4 outputs  : /home/heet18/Downloads/Research Papers/trust_fin_extra_phase/artifacts/phase4
PASS: Phase-2.1 training policy preserved; Phase-4 trains nothing.


## 4. Load train + validation (Phase-4 **never** trains, but validation is the calibration split)

Protected keys are wrapped so they cannot be resolved until the freeze exists **and** the
second opt-in key `TRUSTFIN4_PROTECTED=1` is present. This is asserted, not documented.

In [5]:
PROTECTED_KEYS = {"finqa_test", "fiqa_test", "finben_fomc_test", "fintrust_registry"}
_PROTECTED_STATE = {"opened": False, "opened_keys": []}

def resolve_artifact(key: str) -> Path:
    """Resolve a *training-pool* registry artifact (train / validation / audits)."""
    if key in PROTECTED_KEYS:
        raise RuntimeError(
            f"Protected artifact '{key}' cannot be resolved through resolve_artifact(); "
            "use resolve_protected_artifact() in stage P4-E8."
        )
    entry = REGISTRY["artifacts"][key]
    p = Path(entry["path"])
    if not p.exists():
        raise FileNotFoundError(f"Artifact missing: {key} -> {p}")
    if entry.get("sha256"):
        assert sha256_file(p) == entry["sha256"], f"Hash mismatch for {key}"
    return p

def resolve_protected_artifact(key: str) -> Path:
    """Two-key gate: Phase-4 must be FROZEN and the run must opt in explicitly."""
    assert key in PROTECTED_KEYS, f"{key} is not a protected key"
    if not PHASE4_FREEZE_PATH.exists():
        raise RuntimeError("Refusing to open protected tests: phase4_freeze.json does not exist.")
    freeze = read_json(PHASE4_FREEZE_PATH)
    if freeze.get("status") != "FROZEN":
        raise RuntimeError(f"Refusing to open protected tests: freeze status={freeze.get('status')!r}.")
    if not PROTECTED_OPT_IN:
        raise RuntimeError(
            "Refusing to open protected tests: set TRUSTFIN4_PROTECTED=1 to opt in "
            "(two-key gate, PLAN.md §6.1)."
        )
    entry = REGISTRY["artifacts"][key]
    p = Path(entry["path"])
    if not p.exists():
        raise FileNotFoundError(f"Protected artifact missing: {key} -> {p}")
    if entry.get("sha256"):
        assert sha256_file(p) == entry["sha256"], f"Hash mismatch for {key}"
    _PROTECTED_STATE["opened"] = True
    if key not in _PROTECTED_STATE["opened_keys"]:
        _PROTECTED_STATE["opened_keys"].append(key)
    print(f"[protected] opened {key} -> {p} (freeze={freeze.get('frozen_at_utc')})")
    return p

TRAIN_PATH = resolve_artifact(PHASE3_DATA_CONFIG["train_artifact"])
VAL_PATH = resolve_artifact(PHASE3_DATA_CONFIG["validation_artifact"])
train_records = read_jsonl(TRAIN_PATH)
val_records = read_jsonl(VAL_PATH)

assert all(r["dataset"] in {"FinQA", "FiQA"} for r in train_records)
assert all(r["dataset"] in {"FinQA", "FiQA"} for r in val_records)
assert all(not r.get("metadata", {}).get("evaluation_only", False) for r in train_records)
assert all(not r.get("metadata", {}).get("evaluation_only", False) for r in val_records)

print("Train      :", len(train_records), Counter(r["dataset"] for r in train_records))
print("Validation :", len(val_records), Counter(r["dataset"] for r in val_records))
print("Protected keys are NOT resolvable in this stage:", sorted(PROTECTED_KEYS))

if RUN_MODE == "full":
    assert len(val_records) == 1000
    assert Counter(r["dataset"] for r in val_records) == {"FinQA": 883, "FiQA": 117}

Train      :

 7073 Counter({'FinQA': 6251, 'FiQA': 822})
Validation : 1000 Counter({'FinQA': 883, 'FiQA': 117})
Protected keys are NOT resolvable in this stage: ['finben_fomc_test', 'finqa_test', 'fintrust_registry', 'fiqa_test']


## 5. Utilities: prompts, parsers, official FinQA executor (identical to Phase-3)

In [6]:
def cap_by_dataset(records, cap_per_dataset=None):
    if cap_per_dataset is None:
        return list(records)
    out = []
    for ds in ["FinQA", "FiQA"]:
        subset = [r for r in records if r["dataset"] == ds]
        rng = random.Random(SEED + (0 if ds == "FinQA" else 1000))
        rng.shuffle(subset)
        out.extend(subset[:cap_per_dataset])
    return out

val_working = cap_by_dataset(val_records, VAL_CAP_PER_DATASET)
val_by_id = {r["record_id"]: r for r in val_working}

TASK_TAG = {"FinQA": "<FINQA>", "FiQA": "<FIQA>"}   # frozen; FOMC gets no task token
PROMPT_VARIANT = {"FinQA": FINQA_PROMPT, "FiQA": FIQA_PROMPT}

def user_content(record: Mapping[str, Any], variant: str = "canonical") -> str:
    ds = record["dataset"]
    instruction = record["instruction"].strip()
    inp = record["input"].strip()
    if ds == "FinQA":
        schema = "Return exactly two fields:\nPROGRAM: <valid FinQA program>\nANSWER: <final answer>"
    elif ds == "FiQA":
        schema = "Return exactly:\nSENTIMENT_SCORE: <number from -1 to +1>"
    else:  # FOMC / other eval-only classification tasks
        schema = ""
    if variant == "canonical":
        return f"{TASK_TAG.get(ds, '')}\n{instruction}\n\n{inp}".lstrip("\n")
    if variant == "explicit_schema":
        return f"{TASK_TAG.get(ds, '')}\n{instruction}\n\n{schema}\n\n{inp}".lstrip("\n")
    if variant == "no_task_token":
        return f"{instruction}\n\n{inp}"
    raise ValueError(f"Unknown prompt variant: {variant}")

def chat_messages(record, variant: str = "canonical") -> List[Dict[str, str]]:
    return [
        {"role": "system", "content": record["system"]},
        {"role": "user", "content": user_content(record, variant)},
    ]

def render_chat(tokenizer, messages, add_generation_prompt=True) -> str:
    kwargs = dict(tokenize=False, add_generation_prompt=add_generation_prompt)
    try:
        return tokenizer.apply_chat_template(messages, enable_thinking=False, **kwargs)
    except TypeError:
        return tokenizer.apply_chat_template(messages, **kwargs)
    except Exception:
        parts = [f"{m['role'].upper()}:\n{m['content']}" for m in messages]
        if add_generation_prompt:
            parts.append("ASSISTANT:\n")
        return "\n\n".join(parts)

# ---------------- parsers (identical semantics to Phase-3) ----------------
THINK_RE = re.compile(r"</think>.*?</think>", flags=re.I | re.S)

def clean_generation(text: str) -> str:
    return THINK_RE.sub("", text or "").strip()

def normalize_program(p: Any) -> str:
    return re.sub(r"\s+", "", str(p or "")).lower()

def has_value(x: Any) -> bool:
    """True for a real value; False for None/NaN/pandas-missing (CSV round-trips)."""
    if x is None:
        return False
    if isinstance(x, float) and math.isnan(x):
        return False
    if x is pd.NA or x is pd.NaT:
        return False
    try:
        if pd.isna(x):
            return False
    except Exception:
        pass
    return True

def parse_number_loose(x: Any) -> Optional[float]:
    if x is None:
        return None
    s = str(x).strip().lower().replace(",", "").replace("$", "")
    if not s or s in {"none", "nan", "n/a"}:
        return None
    percent = s.endswith("%")
    if percent:
        s = s[:-1].strip()
    try:
        v = float(s)
        return v / 100.0 if percent else v
    except Exception:
        return None

def loose_equal(a: Any, b: Any, atol=1e-4, rtol=1e-3) -> bool:
    if str(a).strip().lower() == str(b).strip().lower():
        return True
    na, nb = parse_number_loose(a), parse_number_loose(b)
    if na is None or nb is None:
        return False
    return bool(np.isclose(na, nb, atol=atol, rtol=rtol))

def answer_key(a: Any) -> Any:
    """Hashable grouping key for vote counting (numeric-normalised when possible)."""
    v = parse_number_loose(a)
    if v is None:
        return ("s", str(a).strip().lower())
    return ("n", round(v, 4))

def parse_finqa_output(text: str) -> Dict[str, Any]:
    t = clean_generation(text)
    m_prog = re.search(r"PROGRAM\s*:\s*(.+?)(?=\n\s*ANSWER\s*:|$)", t, flags=re.I | re.S)
    m_ans = re.search(r"ANSWER\s*:\s*([^\n]+)", t, flags=re.I)
    program = m_prog.group(1).strip() if m_prog else None
    answer = m_ans.group(1).strip() if m_ans else None
    return {"program": program, "answer": answer, "parse_success": bool(program and answer)}

def parse_fiqa_output(text: str) -> Dict[str, Any]:
    t = clean_generation(text)
    m = re.search(r"SENTIMENT_SCORE\s*:\s*([-+]?(?:\d+(?:\.\d*)?|\.\d+)(?:[eE][-+]?\d+)?)", t, flags=re.I)
    val = None
    if m:
        try:
            val = float(m.group(1))
        except Exception:
            val = None
    if val is None:
        for s in re.findall(r"[-+]?(?:\d+(?:\.\d*)?|\.\d+)", t):
            try:
                z = float(s)
                if -1.0 <= z <= 1.0:
                    val = z
                    break
            except Exception:
                pass
    return {"score": val, "parse_success": val is not None}

FOMC_LABELS = {"hawkish", "dovish", "neutral"}
def parse_fomc_output(text: str) -> Dict[str, Any]:
    t = clean_generation(text).lower()
    for lab in ("hawkish", "dovish", "neutral"):
        if re.search(rf"\b{lab}\b", t):
            return {"label": lab, "parse_success": True}
    return {"label": None, "parse_success": False}

def max_new_tokens_for(dataset: str) -> int:
    if dataset == "FinQA":
        return FINQA_MAX_NEW_TOKENS
    if dataset == "FiQA":
        return FIQA_MAX_NEW_TOKENS
    return OTHER_MAX_NEW_TOKENS

# ---------------- official FinQA executor ----------------
FINQA_EVALUATOR_PATH = Path(PHASE4_DATA_CONFIG["official_evaluators"]["FinQA_local"])
assert FINQA_EVALUATOR_PATH.exists(), f"Official evaluator missing: {FINQA_EVALUATOR_PATH}"
assert sha256_file(FINQA_EVALUATOR_PATH) == PHASE4_DATA_CONFIG["official_evaluators"]["FinQA_sha256"]
FINQA_OFFICIAL = load_module_from_path("finqa_official_phase4", FINQA_EVALUATOR_PATH)

SOURCE_FINGERPRINTS = read_json(Path(REGISTRY["source_fingerprints"]))
_FINQA_DEV = Path(SOURCE_FINGERPRINTS["FinQA"]["dev"]["path"])
_FINQA_TEST_RAW = Path(SOURCE_FINGERPRINTS["FinQA"]["test"]["path"])
FINQA_TABLES = {
    "validation": {str(it["id"]): it["table"] for it in read_json(_FINQA_DEV)},
}
print("FinQA validation tables:", len(FINQA_TABLES["validation"]))

def finqa_tables_for(split: str) -> Dict[str, Any]:
    if split not in FINQA_TABLES:
        raw = {"validation": _FINQA_DEV, "test": _FINQA_TEST_RAW}[split]
        FINQA_TABLES[split] = {str(it["id"]): it["table"] for it in read_json(raw)}
    return FINQA_TABLES[split]

def execute_finqa_program(program: Optional[str], source_id: str,
                          split: str = "validation") -> Tuple[bool, Any]:
    if not program:
        return False, None
    table = finqa_tables_for(split).get(str(source_id))
    if table is None:
        return False, None
    try:
        tokens = FINQA_OFFICIAL.program_tokenization(program)
        invalid_flag, executed = FINQA_OFFICIAL.eval_program(tokens, table)
        return bool(invalid_flag == 0), executed
    except Exception:
        return False, None

print("Official evaluator:", FINQA_EVALUATOR_PATH)
print("Prompt policy      : FinQA=%s | FiQA=%s" % (FINQA_PROMPT, FIQA_PROMPT))
print("Working validation :", Counter(r["dataset"] for r in val_working))

FinQA validation tables: 883
Official evaluator: /home/heet18/Downloads/Research Papers/data/raw/finqa_official_evaluator/evaluate.py
Prompt policy      : FinQA=canonical | FiQA=explicit_schema
Working validation : Counter({'FinQA': 883, 'FiQA': 117})


## 6. Model loading — the frozen E5 backbone (nothing is trained in Phase-4)

In [7]:
def clear_memory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()

def compute_dtype():
    if torch.cuda.is_available() and torch.cuda.is_bf16_supported():
        return torch.bfloat16
    return torch.float16

def quantization_config():
    if not USE_4BIT:
        return None
    return BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type=BNB_QUANT_TYPE,
        bnb_4bit_use_double_quant=BNB_DOUBLE_QUANT,
        bnb_4bit_compute_dtype=compute_dtype(),
    )

def load_tokenizer(model_id: str):
    tok = AutoTokenizer.from_pretrained(model_id, trust_remote_code=False)
    if tok.pad_token_id is None:
        tok.pad_token = tok.eos_token
    tok.padding_side = "right"
    tok.truncation_side = "left"
    return tok

def load_base_model(model_id: str):
    kwargs = dict(trust_remote_code=False, device_map="auto")
    if USE_4BIT:
        kwargs["quantization_config"] = quantization_config()
    else:
        kwargs["torch_dtype"] = compute_dtype()
    model = AutoModelForCausalLM.from_pretrained(model_id, **kwargs)
    model.config.use_cache = True
    return model

MODEL = None
TOKENIZER = None

def get_model():
    """Load the frozen E5 backbone once and reuse it across stages."""
    global MODEL, TOKENIZER
    if MODEL is None:
        clear_memory()
        TOKENIZER = load_tokenizer(BASE_MODEL_ID)
        MODEL = load_base_model(BASE_MODEL_ID)
        MODEL = PeftModel.from_pretrained(MODEL, str(BACKBONE_ADAPTER))
        MODEL.eval()
        if torch.cuda.is_available():
            print(f"[mem] E5 loaded: allocated={torch.cuda.memory_allocated()/2**30:.2f} GiB "
                  f"free={torch.cuda.mem_get_info()[0]/2**30:.2f} GiB", flush=True)
    return MODEL, TOKENIZER

def release_model():
    global MODEL, TOKENIZER
    MODEL, TOKENIZER = None, None
    clear_memory()
    if torch.cuda.is_available():
        print(f"[mem] released: free={torch.cuda.mem_get_info()[0]/2**30:.2f} GiB", flush=True)

## 7. EVG — execution-verified generation

`generate_n` samples *n* candidates from the **frozen** adapter in one call (chunked to stay
inside 6 GB). `evg_select` implements the five pre-declared selection rules, which form the
C1 ablation:

| rule | uses execution? | uses sampling? | final answer emitted |
|---|---|---|---|
| `c0_print` | no | candidate 0 only | model's own `ANSWER:` line (Phase-3 protocol) |
| `c0_exec` | yes | candidate 0 only | executor's result for that program |
| `best_of_n` | no | yes | `ANSWER:` of the highest-logprob sample |
| `self_consistency` | no | yes | majority `ANSWER:` over the samples |
| **`exec_verified`** ⭐ | **yes** | **yes** | **majority *executed* answer** (C1) |

The *frozen Phase-3 greedy* baseline (`greedy_print` / `greedy_exec`, read from
`../artifacts/phase3/predictions/E5.csv` in P4-E1) is reported alongside as `n = 1`, source
`phase3_greedy` — it is a different set of generations than candidate 0 of a sampled pass,
and the two are never conflated in a table.

Ties are always broken by mean token log-probability. No gold program and no gold answer is
consulted by any rule — only the input table, which is part of the model's own prompt.

In [8]:
@torch.inference_mode()
def _sample_chunk(model, tokenizer, enc, n_chunk, max_new, temperature, top_p):
    out = model.generate(
        **enc,
        max_new_tokens=max_new,
        do_sample=True,
        num_return_sequences=n_chunk,
        temperature=temperature,
        top_p=top_p,
        min_new_tokens=1,
        return_dict_in_generate=True,
        output_scores=True,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )
    prompt_len = enc["input_ids"].shape[1]
    seqs = out.sequences[:, prompt_len:]
    texts = tokenizer.batch_decode(seqs, skip_special_tokens=True)
    eos_id = tokenizer.eos_token_id
    logps = []
    try:
        trans = model.compute_transition_scores(out.sequences, out.scores, normalize_logits=True)
        TL = int(trans.shape[1])
        for i in range(seqs.shape[0]):
            ids_row = seqs[i]
            n_valid = int(ids_row.shape[0])
            e = (ids_row == eos_id).nonzero(as_tuple=True)[0]
            if len(e):
                n_valid = int(e[0]) + 1
            n_valid = min(n_valid, TL)
            vals = trans[i, :n_valid].detach().float().cpu().numpy()
            vals = vals[np.isfinite(vals)]
            logps.append(float(np.mean(vals)) if len(vals) else float("nan"))
    except Exception:
        logps = [float("nan")] * len(texts)
    return texts, logps

def generate_candidates(model, tokenizer, record, variant: str, n: int,
                        temperature: float, top_p: float) -> List[Dict[str, Any]]:
    """Sample n candidates, chunked so the 6 GB card never sees n full logit mats."""
    messages = chat_messages(record, variant=variant)
    prompt_text = render_chat(tokenizer, messages, add_generation_prompt=True)
    enc = tokenizer(prompt_text, return_tensors="pt", truncation=True, max_length=MAX_SEQ_LENGTH)
    device = next(model.parameters()).device
    enc = {k: v.to(device) for k, v in enc.items()}
    max_new = max_new_tokens_for(record["dataset"])

    texts, logps = [], []
    chunk = 4
    t0 = time.perf_counter()
    for start in range(0, n, chunk):
        c = min(chunk, n - start)
        try:
            t, l = _sample_chunk(model, tokenizer, enc, c, max_new, temperature, top_p)
        except torch.cuda.OutOfMemoryError:
            clear_memory()
            t, l = _sample_chunk(model, tokenizer, enc, 1, max_new, temperature, top_p)
        texts.extend(t)
        logps.extend(l)
    elapsed = time.perf_counter() - t0
    return [
        {"candidate_index": i, "raw_output": clean_generation(texts[i]),
         "mean_logprob": float(logps[i]),
         "confidence": float(np.exp(np.clip(logps[i], -50, 0))) if np.isfinite(logps[i]) else float("nan"),
         "latency_seconds": float(elapsed / max(len(texts), 1))}
        for i in range(len(texts))
    ]

def enrich_candidates(cands: List[Dict[str, Any]], record: Mapping[str, Any],
                      split: str = "validation") -> List[Dict[str, Any]]:
    """Attach parse + official-execution results to every sampled candidate."""
    out = []
    for c in cands:
        ds = record["dataset"]
        row = dict(c)
        if ds == "FinQA":
            parsed = parse_finqa_output(c["raw_output"])
            executable, executed = execute_finqa_program(parsed["program"], str(record["source_id"]), split)
            row.update({
                "program": parsed["program"], "parsed_answer": parsed["answer"],
                "parse_success": parsed["parse_success"],
                "program_executable": bool(executable), "executed_answer": executed,
            })
        elif ds == "FiQA":
            parsed = parse_fiqa_output(c["raw_output"])
            row.update({"program": None, "parsed_answer": None, "score": parsed["score"],
                        "parse_success": parsed["parse_success"],
                        "program_executable": False, "executed_answer": None})
        else:
            parsed = parse_fomc_output(c["raw_output"])
            row.update({"program": None, "parsed_answer": parsed["label"],
                        "parse_success": parsed["parse_success"],
                        "program_executable": False, "executed_answer": None})
        out.append(row)
    return out

# ---------------- selection rules (C1 ablation) ----------------
def _vote(groups: Dict[Any, List[Dict[str, Any]]]) -> Tuple[List[Dict[str, Any]], float, int]:
    """Largest group wins; ties broken by mean log-probability. Returns (members, agreement, n_pool)."""
    n_pool = sum(len(v) for v in groups.values())
    if not groups:
        return [], float("nan"), n_pool
    def key(kv):
        members = kv[1]
        lps = [m["mean_logprob"] for m in members if np.isfinite(m["mean_logprob"])]
        mean_lp = float(np.mean(lps)) if lps else -1e9
        return (len(members), mean_lp)
    best_key, best_members = max(groups.items(), key=key)
    agreement = len(best_members) / n_pool if n_pool else float("nan")
    rep = max(best_members,
              key=lambda m: m["mean_logprob"] if np.isfinite(m["mean_logprob"]) else -1e9)
    return best_members, agreement, len(best_members)

def evg_select(cands: List[Dict[str, Any]], rule: str) -> Dict[str, Any]:
    assert cands, "no candidates"
    usable = [c for c in cands if c.get("parse_success")]
    pool = usable if usable else list(cands)

    if rule == "c0_print":
        rep, agreement = cands[0], 1.0
        final = rep.get("parsed_answer")
        answer_source = "model_text"
        groups_used = pool
    elif rule == "c0_exec":
        rep = cands[0]
        final = rep.get("executed_answer") if rep.get("program_executable") else rep.get("parsed_answer")
        answer_source = "executor" if rep.get("program_executable") else "model_text"
        agreement, groups_used = 1.0, pool
        if not has_value(final):
            final, answer_source = rep.get("parsed_answer"), "model_text"
    elif rule == "best_of_n":
        rep = max(pool, key=lambda m: m["mean_logprob"] if np.isfinite(m["mean_logprob"]) else -1e9)
        final = rep.get("parsed_answer")
        answer_source = "model_text"
        groups = defaultdict(list)
        for c in pool:
            groups[answer_key(c.get("parsed_answer"))].append(c)
        members, agreement, _ = _vote(groups)
        groups_used = pool
    elif rule == "self_consistency":
        groups = defaultdict(list)
        for c in pool:
            groups[answer_key(c.get("parsed_answer"))].append(c)
        members, agreement, _ = _vote(groups)
        rep = max(members, key=lambda m: m["mean_logprob"]
                  if np.isfinite(m["mean_logprob"]) else -1e9) if members else pool[0]
        final = rep.get("parsed_answer")
        answer_source = "model_text"
        groups_used = pool
    elif rule == "exec_verified":
        ex = [c for c in cands if c.get("program_executable") and has_value(c.get("executed_answer"))]
        if ex:
            groups = defaultdict(list)
            for c in ex:
                groups[answer_key(c["executed_answer"])].append(c)
            members, agreement, _ = _vote(groups)
            rep = max(members, key=lambda m: m["mean_logprob"]
                      if np.isfinite(m["mean_logprob"]) else -1e9)
            final = rep.get("executed_answer")
            answer_source = "executor"
            groups_used = ex
        else:  # no candidate executed: degrade to best-of-n, still report it
            rep = max(pool, key=lambda m: m["mean_logprob"]
                      if np.isfinite(m["mean_logprob"]) else -1e9)
            final = rep.get("parsed_answer")
            answer_source = "model_text_fallback"
            agreement, groups_used = 0.0, pool
    else:
        raise ValueError(rule)

    n_usable = len(pool)
    executable_rate = np.mean([bool(c.get("program_executable")) for c in cands]) if cands else float("nan")
    return {
        "final_answer": final,
        "answer_source": answer_source,
        "selected_program": rep.get("program"),
        "agreement": float(agreement),
        "n_usable": n_usable,
        "n_unique_groups": len({answer_key(c.get("parsed_answer")) for c in groups_used}) if groups_used else 0,
        "executable_rate": float(executable_rate),
        "selected_logprob": float(rep.get("mean_logprob", float("nan"))),
        "selected_confidence": float(rep.get("confidence", float("nan"))),
    }

SELECTION_RULES = ["c0_print", "c0_exec", "best_of_n", "self_consistency", "exec_verified"]
print("Selection rules:", SELECTION_RULES)

Selection rules: ['c0_print', 'c0_exec', 'best_of_n', 'self_consistency', 'exec_verified']


In [9]:
def loose_equal_series(a: pd.Series, b: pd.Series) -> pd.Series:
    return pd.Series([loose_equal(x, y) for x, y in zip(a.fillna(""), b.fillna(""))],
                     index=a.index)

## 8. P4-E1 — baseline table and calibrator benchmark from the frozen Phase-3 predictions

Nothing is re-generated here. We read `../artifacts/phase3/predictions/E5.csv` (read-only),
assert it still reproduces the frozen numbers, and build the *printed-answer* and
*executor* baselines that every Phase-4 result must be compared against.

In [10]:
E5_PRED_PATH = P3_ROOT / "predictions" / "E5.csv"
assert E5_PRED_PATH.exists(), f"Missing frozen E5 predictions: {E5_PRED_PATH}"

if "P4-E1" in RUN_STAGES:
    if (METRIC_DIR / "P4-E1.json").exists() and not OVERWRITE:
        print("P4-E1: using cached metrics.")
        P4E1 = read_json(METRIC_DIR / "P4-E1.json")
    else:
        e5 = pd.read_csv(E5_PRED_PATH)
        e5 = e5[e5["experiment"] == "E5"].copy()
        # keep only records that are in this run's working validation set
        e5 = e5[e5["record_id"].isin(val_by_id)].copy()
        assert len(e5) == len(val_working), (len(e5), len(val_working))

        f = e5[e5["dataset"] == "FinQA"].copy()
        fi = e5[e5["dataset"] == "FiQA"].copy()

        P4E1 = {
            "FinQA": {
                "n": int(len(f)),
                "printed_answer_accuracy": float(f["answer_correct"].mean()),   # greedy_print
                "execution_accuracy": float(f["execution_correct"].mean()),      # greedy_exec
                "program_accuracy": float(f["program_correct"].mean()),
                "parse_success": float(f["parse_success"].mean()),
                "answer_program_consistency": float(loose_equal_series(f["pred_answer"], f["executed_answer"]).mean()),
                "confidence_roc_auc_execution": float(roc_auc_score(f["execution_correct"].astype(int), f["confidence"]))
                    if f["execution_correct"].nunique() > 1 else float("nan"),
            },
            "FiQA": {
                "n": int(len(fi)),
                "mae": float(fi["absolute_error"].mean()),
                "r2": float(r2_score(fi["gold_score"], np.clip(fi["pred_score"], -1, 1))),
                "pearson_r": float(pearsonr(fi["gold_score"], np.clip(fi["pred_score"], -1, 1)).statistic),
                "confidence_pearson_neg_error": float(pearsonr(
                    fi["confidence"], -fi["absolute_error"]).statistic),
                "answer_program_consistency": 1.0,
            },
        }
        # cross-check against the frozen Phase-3 manifest (a drift here invalidates Phase-4).
        # Smoke mode subsamples validation, so the equality only holds on the full split.
        if RUN_MODE == "full":
            w = PHASE3_FREEZE["winner"]
            assert abs(P4E1["FinQA"]["execution_accuracy"] - w["finqa_execution_accuracy"]) < 1e-9, \
                f"FinQA exec drift: {P4E1['FinQA']['execution_accuracy']} vs {w['finqa_execution_accuracy']}"
            assert abs(P4E1["FiQA"]["mae"] - w["fiqa_mae"]) < 1e-9, \
                f"FiQA MAE drift: {P4E1['FiQA']['mae']} vs {w['fiqa_mae']}"
        write_json(METRIC_DIR / "P4-E1.json", P4E1)
        print(json.dumps(P4E1, indent=2))

    print("P4-E1 baseline locked: printed-answer %.4f vs executor %.4f (FinQA execution)"
          % (P4E1["FinQA"]["printed_answer_accuracy"], P4E1["FinQA"]["execution_accuracy"]))
else:
    P4E1 = read_json(METRIC_DIR / "P4-E1.json") if (METRIC_DIR / "P4-E1.json").exists() else {}

P4-E1: using cached metrics.
P4-E1 baseline locked: printed-answer 0.3760 vs executor 0.6750 (FinQA execution)


## 9. P4-E2 — EVG sampling pass on the FinQA validation set

One GPU pass, *n* = `EVG_N` candidates per record (n = 2, 4 are evaluated as **prefixes** of
the same samples, so `n ∈ {2,4,8}` costs one pass, not three). Candidates are appended to a
partial JSONL after every record, so an interrupted run resumes instead of restarting.

In [11]:
def sample_split(exp_id: str, records: Sequence[Mapping[str, Any]], split: str,
                 n: int, temperature: float, top_p: float) -> pd.DataFrame:
    """Sample n candidates for every record with resume support."""
    final_path = PRED_DIR / f"{exp_id}_candidates.csv"
    partial_path = PRED_DIR / f"{exp_id}_candidates.partial.jsonl"

    if final_path.exists() and not OVERWRITE:
        print(f"{exp_id}: using cached candidates ({final_path.name}).")
        return pd.read_csv(final_path)

    done: Dict[str, int] = {}
    if partial_path.exists():
        for line in partial_path.open("r", encoding="utf-8"):
            if line.strip():
                r = json.loads(line)
                done[r["record_id"]] = done.get(r["record_id"], 0) + 1
        print(f"{exp_id}: resuming, {len(done)} records already sampled.")

    todo = [r for r in records if done.get(r["record_id"], 0) < n]
    print(f"{exp_id}: {len(todo)}/{len(records)} records to sample (n={n}, split={split})")

    model, tokenizer = get_model()
    t_start = time.perf_counter()
    with partial_path.open("a", encoding="utf-8") as fh:
        for i, rec in enumerate(todo, 1):
            variant = PROMPT_VARIANT.get(rec["dataset"], "canonical")
            cands = generate_candidates(model, tokenizer, rec, variant, n, temperature, top_p)
            cands = enrich_candidates(cands, rec, split=split)
            for c in cands:
                row = {
                    "experiment": exp_id, "record_id": rec["record_id"],
                    "source_id": rec["source_id"], "dataset": rec["dataset"],
                    "split": split, **c,
                }
                fh.write(json.dumps(row, ensure_ascii=False, default=str) + "\n")
            fh.flush()
            if i % 25 == 0 or i == len(todo):
                el = time.perf_counter() - t_start
                print(f"{exp_id}: {i}/{len(todo)} records | {el/60:.1f} min | "
                      f"{el/max(i,1):.2f} s/record", flush=True)

    rows = [json.loads(l) for l in partial_path.open("r", encoding="utf-8") if l.strip()]
    df = pd.DataFrame(rows)
    counts = df.groupby("record_id").size()
    incomplete = counts[counts < n]
    if len(incomplete):
        raise RuntimeError(f"{exp_id}: {len(incomplete)} records have fewer than {n} candidates; "
                           "re-run to finish the pass.")
    df.to_csv(final_path, index=False)
    partial_path.unlink(missing_ok=True)
    print(f"{exp_id}: wrote {final_path} ({len(df)} candidate rows, "
          f"{df['record_id'].nunique()} records)")
    return df

if "P4-E2" in RUN_STAGES:
    finqa_val = [r for r in val_working if r["dataset"] == "FinQA"]
    P4E2_cands = sample_split("P4-E2", finqa_val, "validation", EVG_N, EVG_TEMPERATURE, EVG_TOP_P)
else:
    p = PRED_DIR / "P4-E2_candidates.csv"
    P4E2_cands = pd.read_csv(p) if p.exists() else pd.DataFrame()
    print("P4-E2 skipped; candidates on disk:", len(P4E2_cands))

P4-E2 skipped; candidates on disk: 7064


## 10. P4-E3 — selection ablation (the C1 evidence)

Applies all five selection rules to the **same** sampled candidates, so the only variable is
the selection rule. Also evaluates `n ∈ {2, 4, 8}` by truncating each record's candidate list,
which is why P4-E2 samples `EVG_N` once.

In [12]:
def score_finqa_selection(records: Sequence[Mapping[str, Any]],
                          cand_df: pd.DataFrame, exp_id: str,
                          ns: Sequence[int], split: str = "validation") -> pd.DataFrame:
    # The executor is table-driven: scoring test records against the validation table
    # directory would silently mark every program non-executable. Guard it.
    if "split" in cand_df.columns and len(cand_df):
        got = set(map(str, cand_df["split"].unique()))
        assert got == {split}, f"{exp_id}: candidates split={got} but scoring split={split!r}"
    by_rec = {str(rid): g.sort_values("candidate_index").to_dict("records")
              for rid, g in cand_df.groupby("record_id")}
    rows = []
    for rec in records:
        cands_all = by_rec.get(str(rec["record_id"]))
        if not cands_all:
            continue
        for n in ns:
            cands = cands_all[:n]
            for rule in SELECTION_RULES:
                sel = evg_select(cands, rule)
                executable, executed = execute_finqa_program(
                    sel["selected_program"], str(rec["source_id"]), split)
                final = sel["final_answer"]
                rows.append({
                    "experiment": exp_id, "n_candidates": n, "rule": rule,
                    "record_id": rec["record_id"], "source_id": rec["source_id"],
                    "final_answer": final, "answer_source": sel["answer_source"],
                    "selected_program": sel["selected_program"],
                    "agreement": sel["agreement"], "executable_rate": sel["executable_rate"],
                    "n_usable": sel["n_usable"], "selected_logprob": sel["selected_logprob"],
                    "selected_confidence": sel["selected_confidence"],
                    "final_answer_correct": bool(loose_equal(final, rec["target_answer"])),
                    "program_correct": normalize_program(sel["selected_program"]) == normalize_program(rec["gold_program"]),
                    "program_executable": bool(executable),
                    "executed_answer": executed,
                    "execution_correct": bool(executable and loose_equal(executed, rec["target_answer"])),
                    "gold_answer": rec["target_answer"],
                })
    return pd.DataFrame(rows)

def summarise_finqa(df: pd.DataFrame) -> pd.DataFrame:
    g = df.groupby(["n_candidates", "rule"]).agg(
        n=("record_id", "size"),
        answer_accuracy=("final_answer_correct", "mean"),
        execution_accuracy=("execution_correct", "mean"),
        program_accuracy=("program_correct", "mean"),
        invalid_program_rate=("program_executable", lambda s: float((~s).mean())),
        mean_agreement=("agreement", "mean"),
    ).reset_index()
    g["selective_risk_1_acc"] = 1.0 - g["answer_accuracy"]
    return g

NS_EVAL = sorted({n for n in (1, 2, 4, 8) if n <= EVG_N})
if "P4-E3" in RUN_STAGES and len(P4E2_cands):
    finqa_val = [r for r in val_working if r["dataset"] == "FinQA"]
    if (ANALYSIS_DIR / "P4-E3_selection_ablation.csv").exists() and not OVERWRITE:
        print("P4-E3: using cached ablation.")
        P4E3 = pd.read_csv(ANALYSIS_DIR / "P4-E3_selection_ablation.csv")
    else:
        detail = score_finqa_selection(finqa_val, P4E2_cands, "P4-E3", NS_EVAL)
        detail.to_csv(PRED_DIR / "P4-E3.csv", index=False)
        P4E3 = summarise_finqa(detail)
        P4E3.to_csv(ANALYSIS_DIR / "P4-E3_selection_ablation.csv", index=False)
        write_json(METRIC_DIR / "P4-E3.json", P4E3.to_dict("records"))
        print(P4E3.to_string(index=False))
else:
    p = ANALYSIS_DIR / "P4-E3_selection_ablation.csv"
    P4E3 = pd.read_csv(p) if p.exists() else pd.DataFrame()
    print("P4-E3 skipped; rows on disk:", len(P4E3))

P4-E3 skipped; rows on disk: 20


## 11. P4-E3b — FiQA self-consistency (sampling applied to the regression task)

EVG's executor does not exist for sentiment, so the sampling analogue is **median-of-n**
over `SENTIMENT_SCORE` samples. Greedy is the frozen Phase-3 number; the three aggregators
are compared on the identical sample set.

In [13]:
def summarise_fiqa_sampling(cand_df: pd.DataFrame, exp_id: str,
                            base_rows: pd.DataFrame) -> pd.DataFrame:
    """base_rows: one row per record with gold_score + greedy confidence."""
    rows = []
    if not len(cand_df):
        return pd.DataFrame(), pd.DataFrame()
    grouped = {str(rid): g.sort_values("candidate_index").to_dict("records")
               for rid, g in cand_df.groupby("record_id")}
    for _, b in base_rows.iterrows():
        rid = b["record_id"]
        cands = grouped.get(str(rid)) or []
        if not cands:
            continue
        usable = [c for c in cands if has_value(c.get("score"))]
        scores = [float(c["score"]) for c in usable]
        confs = [float(c["confidence"]) for c in usable if has_value(c.get("confidence"))]
        gold = float(b["gold_score"])
        greedy = float(b["pred_score"])
        sample_std = float(np.std(scores)) if len(scores) > 1 else 0.0
        mean_conf = float(np.mean(confs)) if confs else float(b["confidence"])
        variants = {
            "greedy": greedy,
            "median_n": float(np.median(scores)) if scores else greedy,
            "mean_n": float(np.mean(scores)) if scores else greedy,
            "conf_weighted_mean": (
                float(np.average(scores, weights=[max(float(c["confidence"]), 1e-6)
                                                  for c in usable if has_value(c.get("confidence"))]))
                if scores and confs else greedy
            ),
        }
        for name, pred in variants.items():
            pred_c = float(np.clip(pred, -1, 1))
            rows.append({
                "experiment": exp_id, "record_id": rid, "aggregator": name,
                "pred_score": pred_c, "gold_score": gold,
                "absolute_error": abs(pred_c - gold),
                "correct_tol_0.10": bool(abs(pred_c - gold) <= 0.10),
                "confidence": mean_conf, "mean_logprob": float(b["mean_logprob"]),
                "sample_std": sample_std, "n_samples": len(scores),
            })
    df = pd.DataFrame(rows)
    agg = df.groupby("aggregator").agg(
        n=("record_id", "size"), mae=("absolute_error", "mean"),
        acc_tol_0_10=("correct_tol_0.10", "mean"),
    ).reset_index()
    r2s, prs = [], []
    for name in agg["aggregator"]:
        sub = df[df["aggregator"] == name]
        r2s.append(r2_score(sub["gold_score"], sub["pred_score"]) if len(sub) >= 2 else float("nan"))
        prs.append(float(pearsonr(sub["gold_score"], sub["pred_score"]).statistic)
                   if len(sub) >= 2 and sub["pred_score"].std() > 0 else float("nan"))
    agg["r2"] = r2s
    agg["pearson_r"] = prs
    return df, agg

if "P4-E3b" in RUN_STAGES:
    fiqa_val = [r for r in val_working if r["dataset"] == "FiQA"]
    if (ANALYSIS_DIR / "P4-E3b_fiqa_sampling.csv").exists() and not OVERWRITE:
        print("P4-E3b: using cached aggregation.")
        P4E3b_detail = pd.read_csv(PRED_DIR / "P4-E3b.csv")
        P4E3b = pd.read_csv(ANALYSIS_DIR / "P4-E3b_fiqa_sampling.csv")
    else:
        P4E3b_cands = sample_split("P4-E3b", fiqa_val, "validation", EVG_N,
                                   EVG_TEMPERATURE, EVG_TOP_P)
        e5 = pd.read_csv(E5_PRED_PATH)
        base = e5[(e5["experiment"] == "E5") & (e5["dataset"] == "FiQA")
                  & (e5["record_id"].isin(val_by_id))].copy()
        P4E3b_detail, P4E3b = summarise_fiqa_sampling(P4E3b_cands, "P4-E3b", base)
        P4E3b_detail.to_csv(PRED_DIR / "P4-E3b.csv", index=False)
        P4E3b.to_csv(ANALYSIS_DIR / "P4-E3b_fiqa_sampling.csv", index=False)
        write_json(METRIC_DIR / "P4-E3b.json", P4E3b.to_dict("records"))
        print(P4E3b.to_string(index=False))
    release_model()
else:
    p = ANALYSIS_DIR / "P4-E3b_fiqa_sampling.csv"
    P4E3b = pd.read_csv(p) if p.exists() else pd.DataFrame()
    print("P4-E3b skipped; rows on disk:", len(P4E3b))

P4-E3b skipped; rows on disk: 4


## 12. P4-E4 / P4-E5 / P4-E6 — cross-fitted calibration (the C2 + C3 evidence)

Calibrators are fitted **inside validation only**, with 5-fold cross-fitting: every record's
calibrated probability comes from a model that never saw that record, so the reported
risk-coverage is out-of-fold rather than in-sample. After selection, the winning calibrator is
refit on the whole validation split and frozen.

| stage | method | features | claim tested |
|---|---|---|---|
| P4-E4 | Platt (logistic) | confidence | **C2** — calibration beats raw confidence |
| P4-E5 | isotonic | confidence (univariate) | non-parametric alternative |
| P4-E6 | Platt | confidence + **EVG agreement** + executable rate | **C3** — verification helps abstention |

In [14]:
class PlattCalibrator:
    name = "platt"
    def __init__(self):
        from sklearn.pipeline import Pipeline
        self.pipe = Pipeline([
            ("impute", SimpleImputer(strategy="median")),
            ("scale", StandardScaler()),
            ("clf", LogisticRegression(C=1.0, max_iter=2000)),
        ])
    def fit(self, X, y):
        self.pipe.fit(np.asarray(X, float), np.asarray(y, int))
        return self
    def predict(self, X):
        return self.pipe.predict_proba(np.asarray(X, float))[:, 1]

class IsotonicCalibrator:
    name = "isotonic"
    def __init__(self):
        self.iso = IsotonicRegression(out_of_bounds="clip", y_min=0.0, y_max=1.0)
    def fit(self, X, y):
        X = np.asarray(X, float)
        self._col = 0 if X.ndim == 1 else 0          # univariate on column 0
        imp = np.nanmedian(X[:, self._col])
        xs = np.where(np.isfinite(X[:, self._col]), X[:, self._col], imp)
        self._imp = imp
        self.iso.fit(xs, np.asarray(y, float))
        return self
    def predict(self, X):
        X = np.asarray(X, float)
        xs = X[:, self._col] if X.ndim > 1 else X
        xs = np.where(np.isfinite(xs), xs, self._imp)
        return np.clip(self.iso.predict(xs), 0.0, 1.0)

def make_calibrator(method: str):
    if method == "platt":
        return PlattCalibrator()
    if method == "isotonic":
        return IsotonicCalibrator()
    raise ValueError(method)

def cross_fitted_scores(X, y, method: str, folds: int = CALIB_FOLDS, seed: int = SEED):
    """Out-of-fold calibrated probabilities (each point scored by a model that never saw it)."""
    X = np.asarray(X, float)
    y = np.asarray(y, int)
    oof = np.full(len(y), np.nan, float)
    if len(y) < folds * 2 or len(np.unique(y)) < 2:
        return np.full(len(y), float(np.mean(y))), None
    kf = KFold(n_splits=folds, shuffle=True, random_state=seed)
    for tr, te in kf.split(X):
        if len(np.unique(y[tr])) < 2:
            oof[te] = float(np.mean(y[tr]))
            continue
        cal = make_calibrator(method).fit(X[tr], y[tr])
        oof[te] = cal.predict(X[te])
    final = make_calibrator(method).fit(X, y)      # refit on all of validation (this is what freezes)
    return oof, final

# ---------------- selective prediction ----------------
def selective_metrics(scores, errors, coverage_targets=COVERAGE_TARGETS):
    """errors: 1 = wrong (binary) or a non-negative continuous loss. Higher score = more confident."""
    scores = np.asarray(scores, float)
    errors = np.asarray(errors, float)
    ok = np.isfinite(scores) & np.isfinite(errors)
    scores, errors = scores[ok], errors[ok]
    n = len(scores)
    if n == 0:
        return {"n": 0}
    order = np.argsort(-scores, kind="mergesort")
    e = errors[order]
    cum = np.cumsum(e)
    risks = cum / np.arange(1, n + 1)
    aurc = float(risks.mean())
    e_sorted = np.sort(errors)                    # perfect ranking
    opt_risks = np.cumsum(e_sorted) / np.arange(1, n + 1)
    aurc_opt = float(opt_risks.mean())
    out = {
        "n": int(n),
        "full_risk": float(errors.mean()),
        "aurc": aurc,
        "aurc_optimal": aurc_opt,
        "e_aurc": aurc - aurc_opt,
    }
    for c in coverage_targets:
        k = max(1, int(round(c * n)))
        out[f"risk@cov{c:.2f}"] = float(e[:k].mean())
        out[f"coverage@1.0"] = float(1.0)
    # coverage achievable at frozen risk tolerances
    return out

def brier_nll(y, p):
    y = np.asarray(y, int); p = np.clip(np.asarray(p, float), 1e-6, 1 - 1e-6)
    return {
        "brier": float(brier_score_loss(y, p)),
        "nll": float(-np.mean(y * np.log(p) + (1 - y) * np.log(1 - p))),
        "roc_auc": float(roc_auc_score(y, p)) if len(np.unique(y)) > 1 else float("nan"),
    }

def ece_score(y, p, bins=10):
    y = np.asarray(y, int); p = np.asarray(p, float)
    edges = np.linspace(0, 1, bins + 1)
    tot = 0.0
    for i in range(bins):
        m = (p >= edges[i]) & (p < edges[i + 1] + (1e-9 if i == bins - 1 else 0))
        if m.sum():
            tot += m.sum() / len(y) * abs(y[m].mean() - p[m].mean())
    return float(tot)

### Building the calibration frames (one frame per system under comparison)

In [15]:
FRAMES: Dict[str, Dict[str, Any]] = {}

def add_frame(name, X, y, task, note=""):
    FRAMES[name] = {"X": np.asarray(X, float), "y": np.asarray(y, int),
                    "task": task, "note": note, "n": len(y), "base_risk": float(1 - np.mean(y))}

# ---- FinQA: frozen Phase-3 greedy (two emission rules) ----
_e5 = pd.read_csv(E5_PRED_PATH)
_e5 = _e5[_e5["record_id"].isin(val_by_id)]
_ef = _e5[_e5["dataset"] == "FinQA"].copy()
assert len(_ef) == len([r for r in val_working if r["dataset"] == "FinQA"])

add_frame("finqa_greedy_print",
          _ef[["confidence", "mean_logprob", "program_executable"]].astype(float),
          _ef["answer_correct"].astype(int), "FinQA",
          "frozen Phase-3 greedy, model's own ANSWER line")
add_frame("finqa_greedy_exec",
          _ef[["confidence", "mean_logprob", "program_executable"]].astype(float),
          _ef["execution_correct"].astype(int), "FinQA",
          "frozen Phase-3 greedy, executor answer")

# ---- FinQA: Phase-4 selection rules from the sampled pass ----
if len(P4E2_cands):
    detail_path = PRED_DIR / "P4-E3.csv"
    if detail_path.exists():
        d3 = pd.read_csv(detail_path)
        n_use = int(d3["n_candidates"].max())
        for rule in SELECTION_RULES:
            sub = d3[(d3["n_candidates"] == n_use) & (d3["rule"] == rule)].copy()
            if not len(sub):
                continue
            feats = sub[["selected_confidence", "agreement", "executable_rate"]].astype(float)
            if rule == "exec_verified":
                add_frame("finqa_exec_verified", feats, sub["final_answer_correct"].astype(int),
                          "FinQA", f"EVG n={n_use}, majority executed answer")
            elif rule == "self_consistency":
                add_frame("finqa_self_consistency", feats, sub["final_answer_correct"].astype(int),
                          "FinQA", f"EVG n={n_use}, majority parsed answer")
            elif rule == "c0_exec":
                add_frame("finqa_c0_exec", feats, sub["final_answer_correct"].astype(int),
                          "FinQA", f"EVG n=1 (candidate 0), executor answer")

# ---- FiQA: frozen greedy + EVG median ----
_ifi = _e5[_e5["dataset"] == "FiQA"].copy()
_tol = 0.10   # pre-declared correctness tolerance for a [-1,1] regression target
if len(_ifi):
    err = _ifi["absolute_error"].astype(float)
    add_frame("fiqa_greedy",
              _ifi[["confidence", "mean_logprob"]].astype(float),
              (err <= _tol).astype(int), "FiQA", "frozen Phase-3 greedy, |err| <= 0.10")

p3b = PRED_DIR / "P4-E3b.csv"
if p3b.exists():
    d3b = pd.read_csv(p3b)
    sub = d3b[d3b["aggregator"] == "median_n"].copy()
    if len(sub):
        add_frame("fiqa_median_n",
                  sub[["confidence", "sample_std", "mean_logprob"]].astype(float),
                  (sub["absolute_error"] <= _tol).astype(int), "FiQA",
                  f"EVG median-of-{int(sub['n_samples'].max())}, |err| <= 0.10")

for k, v in FRAMES.items():
    print(f"{k:26s} n={v['n']:4d}  base_risk={v['base_risk']:.3f}  task={v['task']}  ({v['note']})")

finqa_greedy_print         n= 883  base_risk=0.624  task=FinQA  (frozen Phase-3 greedy, model's own ANSWER line)
finqa_greedy_exec          n= 883  base_risk=0.325  task=FinQA  (frozen Phase-3 greedy, executor answer)
finqa_c0_exec              n= 883  base_risk=0.350  task=FinQA  (EVG n=1 (candidate 0), executor answer)
finqa_self_consistency     n= 883  base_risk=0.617  task=FinQA  (EVG n=8, majority parsed answer)
finqa_exec_verified        n= 883  base_risk=0.325  task=FinQA  (EVG n=8, majority executed answer)
fiqa_greedy                n= 117  base_risk=0.453  task=FiQA  (frozen Phase-3 greedy, |err| <= 0.10)
fiqa_median_n              n= 117  base_risk=0.444  task=FiQA  (EVG median-of-8, |err| <= 0.10)


### P4-E4 (Platt) · P4-E5 (isotonic) · P4-E6 (Platt + EVG agreement)

**What each metric can and cannot show (read before interpreting the tables).**

- `brier` / `ece` / `nll` measure *probability quality*. A monotone calibrator fitted on the
  **same** scores it is applied to cannot change their ranking, so it **cannot** improve
  AURC — those metrics are the honest test of C2.
- `aurc` / `e_aurc` measure *ranking*. A single-feature Platt map (P4-E4) is monotone in the
  confidence, so **after it is refitted on all of validation its ranking is identical to
  raw confidence**; AURC can only differ through the cross-fitting folds, where each fold's
  map is fit on different data and therefore reorders points across folds. The out-of-fold
  AURC numbers below are thus a *deployable-selection* diagnostic (they use only the score
  each record would really have got), not evidence that Platt reorders predictions.
- **C3 is the claim that can change ranking.** `platt_conf_full` (P4-E6) adds EVG agreement and
  executable-program rate as extra features, so it is *not* a monotone map of confidence and
  can genuinely reorder records. If C3 holds it must show up as a lower out-of-fold
  `e_aurc` for `platt_conf_full` than for `platt_conf`/`raw_confidence`.
- Consequently `pick_best` (lowest out-of-fold `e_aurc`) will legitimately return
  `raw_confidence` for a frame if nothing reorders it. That is a **negative result for C3 on
  that frame, not a bug**, and it is what the freeze will record.

In [16]:
CALIB_FEATURE_SETS = {
    # stage -> (methods applied to every frame, extra frames restricted to that stage)
    "P4-E4": ["platt_conf"],
    "P4-E5": ["isotonic_conf"],
    "P4-E6": ["platt_conf_full"],
}
FEATURE_COLS = {
    "platt_conf": {"FinQA": [0], "FiQA": [0]},          # column 0 = confidence
    "isotonic_conf": {"FinQA": [0], "FiQA": [0]},
    "platt_conf_full": {"FinQA": [0, 1, 2], "FiQA": [0, 1, 2]},  # + agreement + executable_rate
}

def run_calibration(stage: str) -> pd.DataFrame:
    out_rows = []
    for spec in CALIB_FEATURE_SETS[stage]:
        for fname, fr in FRAMES.items():
            cols = FEATURE_COLS[spec][fr["task"]]
            if spec == "platt_conf_full" and fr["task"] == "FiQA":
                cols = [0, 1]                            # FiQA frames have no agreement feature
            if max(cols) >= fr["X"].shape[1]:
                continue                                 # frame lacks these features
            X = fr["X"][:, cols]
            y = fr["y"]
            method = "isotonic" if spec.startswith("isotonic") else "platt"
            oof, fitted = cross_fitted_scores(X, y, method)
            row = {"stage": stage, "frame": fname, "method": spec, "calibrator": method,
                   "task": fr["task"], "features": ",".join(str(c) for c in cols),
                   "n": len(y), "base_risk": fr["base_risk"]}
            row.update(brier_nll(y, oof))
            row["ece"] = ece_score(y, oof)
            row.update(selective_metrics(oof, 1 - y))
            # raw (uncalibrated) confidence is the control
            raw = fr["X"][:, 0]
            row_raw = {"stage": stage, "frame": fname, "method": "raw_confidence",
                       "calibrator": "none", "task": fr["task"], "features": "0",
                       "n": len(y), "base_risk": fr["base_risk"]}
            row_raw.update(brier_nll(y, np.clip(raw, 1e-6, 1 - 1e-6)))
            row_raw["ece"] = ece_score(y, np.clip(raw, 1e-6, 1 - 1e-6))
            row_raw.update(selective_metrics(raw, 1 - y))
            out_rows.extend([row_raw, row])
            # persist the refit calibrator for the freeze stage
            if stage == "P4-E6" or (stage == "P4-E4" and fname in {"finqa_exec_verified", "fiqa_greedy"}):
                import pickle
                with (MANIFEST_DIR / f"calibrator_{fname}_{method}.pkl").open("wb") as fh:
                    pickle.dump({"features": cols, "calibrator": fitted, "frame": fname,
                                 "method": method, "n": len(y)}, fh)
    df = pd.DataFrame(out_rows).drop_duplicates(subset=["stage", "frame", "method"])
    df.to_csv(ANALYSIS_DIR / f"{stage}_calibration.csv", index=False)
    write_json(METRIC_DIR / f"{stage}.json", df.to_dict("records"))
    return df

CALIB_RESULTS = {}
for stage in ["P4-E4", "P4-E5", "P4-E6"]:
    path = ANALYSIS_DIR / f"{stage}_calibration.csv"
    if stage in RUN_STAGES and (not path.exists() or OVERWRITE):
        print(f"\n=== {stage} ===")
        CALIB_RESULTS[stage] = run_calibration(stage)
        print(CALIB_RESULTS[stage][["frame", "method", "brier", "ece", "aurc", "e_aurc",
                                    "risk@cov0.80"]].to_string(index=False))
    elif path.exists():
        CALIB_RESULTS[stage] = pd.read_csv(path)
        print(f"{stage}: cached ({len(CALIB_RESULTS[stage])} rows)")
    else:
        print(f"{stage}: skipped (not in RUN_STAGES)")

P4-E4: cached (14 rows)
P4-E5: cached (14 rows)
P4-E6: cached (14 rows)


## 13. P4-E7 — selective-prediction curves and the Phase-4 freeze

**Selection is validation-only, and the rule is declared up front:** the Phase-4 method is
`exec_verified` at `n = EVG_N` with the calibrator that attains the lowest out-of-fold AURC
on validation among {raw confidence, Platt(confidence), isotonic(confidence),
Platt(confidence + agreement + executable rate)}. Operating thresholds are then read off at
coverage 0.80 and 0.90 — never at a test-derived risk.

In [17]:
def collect_stage_frames():
    rows = []
    for stage, dfr in CALIB_RESULTS.items():
        if dfr is None or not len(dfr):
            continue
        for _, r in dfr.iterrows():
            rows.append({"stage": stage, **r.to_dict()})
    df = pd.DataFrame(rows)
    # raw_confidence is re-emitted as the control by every stage; keep one row per frame+method
    return df.drop_duplicates(subset=["frame", "method"], keep="first")

if "P4-E7" in RUN_STAGES:
    allc = collect_stage_frames()
    if not len(allc):
        raise RuntimeError("No calibration results found — run P4-E4/P4-E5/P4-E6 first.")
    allc.to_csv(ANALYSIS_DIR / "P4-E7_all_calibration.csv", index=False)

    # ---- pick the Phase-4 method, validation only ----
    finqa_cand = allc[(allc["task"] == "FinQA") & (allc["frame"] == "finqa_exec_verified")]
    fiqa_cand = allc[(allc["task"] == "FiQA")]
    if not len(finqa_cand):
        raise RuntimeError("finqa_exec_verified frame missing — run P4-E2/P4-E3 first.")

    def pick_best(df):
        d = df.sort_values("e_aurc").drop_duplicates(subset=["frame", "method"])
        return d.iloc[0].to_dict()

    best_finqa = pick_best(finqa_cand)
    # FiQA: choose among the available FiQA frames (validation only)
    best_fiqa = pick_best(fiqa_cand) if len(fiqa_cand) else None

    # How each frame is reconstructed on the protected test set (P4-E8e).
    FINQA_FRAME_RULE = {"finqa_exec_verified": "exec_verified",
                        "finqa_self_consistency": "self_consistency",
                        "finqa_c0_exec": "c0_exec"}
    FIQA_FRAME_AGG = {"fiqa_median_n": "median_n", "fiqa_greedy": "greedy"}

    # ---- operating thresholds at the pre-declared coverages ----
    import pickle
    def operating_points(frame_name, method, calibrator_name, task):
        fr = FRAMES[frame_name]
        y = fr["y"]
        fitted = None
        if method == "raw_confidence":
            cols, scores = [0], fr["X"][:, 0]
        else:
            cols = FEATURE_COLS.get(method, {task: [0]})[task]
            if max(cols) >= fr["X"].shape[1]:
                cols = list(range(fr["X"].shape[1]))
            oof, fitted = cross_fitted_scores(fr["X"][:, cols], y, calibrator_name)
            scores = oof if np.isfinite(oof).any() else fr["X"][:, 0]
        s = np.sort(scores)[::-1]
        pts = {}
        for c in FROZEN_COVERAGES:
            k = max(1, int(round(c * len(s))))
            thr = float(s[k - 1])
            sel = scores >= thr
            errs = (1 - y).astype(float)
            pts[f"coverage_{c:.2f}"] = {
                "threshold": thr,
                "achieved_coverage": float(sel.mean()),
                "validation_selective_risk": float(errs[sel].mean()) if sel.any() else float("nan"),
                "validation_full_risk": float(errs.mean()),
            }
        if fitted is not None:
            with (MANIFEST_DIR / f"calibrator_{frame_name}_{calibrator_name}_final.pkl").open("wb") as fh:
                pickle.dump({"features": cols, "calibrator": fitted, "frame": frame_name,
                             "calibrator_name": calibrator_name, "n": len(y)}, fh)
        else:
            write_json(MANIFEST_DIR / f"calibrator_{frame_name}_raw.json",
                       {"features": cols, "frame": frame_name, "method": "raw_confidence"})
        return pts

    finqa_ops = operating_points(best_finqa["frame"], best_finqa["method"],
                                 "isotonic" if str(best_finqa["method"]).startswith("isotonic") else "platt",
                                 "FinQA")
    fiqa_ops = (operating_points(best_fiqa["frame"], best_fiqa["method"],
                                 "isotonic" if str(best_fiqa["method"]).startswith("isotonic") else "platt",
                                 "FiQA") if best_fiqa else {})

    # ---- risk-coverage curves for the figures ----
    def rc_curve(scores, errors, grid=np.linspace(0.05, 1.0, 39)):
        scores = np.asarray(scores, float); errors = np.asarray(errors, float)
        ok = np.isfinite(scores) & np.isfinite(errors)
        scores, errors = scores[ok], errors[ok]
        order = np.argsort(-scores, kind="mergesort")
        e = errors[order]
        n = len(e)
        return [{"coverage": float(c), "risk": float(e[:max(1, int(round(c * n)))].mean())}
                for c in grid]

    curves = {}
    for fname, fr in FRAMES.items():
        X, y = fr["X"], fr["y"]
        method = best_finqa["method"] if fr["task"] == "FinQA" else (
            best_fiqa["method"] if best_fiqa else "platt_conf")
        cal = "isotonic" if str(method).startswith("isotonic") else "platt"
        cols = FEATURE_COLS.get(method, {fr["task"]: [0]})[fr["task"]]
        if max(cols) >= X.shape[1]:
            cols = list(range(X.shape[1]))
        oof, _ = cross_fitted_scores(X[:, cols], y, cal)
        scores = oof if np.isfinite(oof).any() else X[:, 0]
        curves[fname] = {"raw": rc_curve(X[:, 0], 1 - y), "calibrated": rc_curve(scores, 1 - y)}
    write_json(MANIFEST_DIR / "p4_risk_coverage_curves.json", curves)

    freeze = {
        "phase": "Phase-4",
        "status": "FROZEN",
        "method_name": "EVG-CAS",
        "frozen_at_utc": pd.Timestamp.now(tz="UTC").isoformat(),
        "run_mode": RUN_MODE,
        "novelty_claims": {
            "C1": "execution-verified generation: majority vote over officially executed candidate programs",
            "C2": "cross-fitted calibrated abstention fitted on validation only",
            "C3": "EVG agreement improves the calibrator over confidence alone",
        },
        "backbone": {
            "experiment": "E5", "model_id": BASE_MODEL_ID,
            "adapter_path": str(BACKBONE_ADAPTER),
            "phase3_selection_score": PHASE3_FREEZE["winner"]["selection_score"],
            "changed_by_phase4": False,
        },
        "prompt_policy": {"FinQA": FINQA_PROMPT, "FiQA": FIQA_PROMPT, "FOMC": "canonical"},
        "evg": {
            "n": EVG_N, "temperature": EVG_TEMPERATURE, "top_p": EVG_TOP_P,
            "selection_rule": FINQA_FRAME_RULE.get(best_finqa["frame"], "exec_verified"),
            "tie_break": "mean_token_logprob",
            "uses_gold_answer": False, "uses_gold_program": False,
            "uses_only": "the input table, which is part of the model's own prompt",
        },
        "calibration": {
            "split": PHASE4_DATA_CONFIG["calibration_split"],
            "folds": CALIB_FOLDS,
            "selection_rule": "lowest out-of-fold e_AURC on validation among raw/platt/isotonic",
            "finqa": {"frame": best_finqa["frame"], "method": best_finqa["method"],
                      "selection_rule": FINQA_FRAME_RULE.get(best_finqa["frame"]),
                      "n_candidates": int(EVG_N),
                      "e_aurc": best_finqa["e_aurc"], "aurc": best_finqa["aurc"],
                      "base_risk": best_finqa["base_risk"]},
            "fiqa": ({"frame": best_fiqa["frame"], "method": best_fiqa["method"],
                      "aggregator": FIQA_FRAME_AGG.get(best_fiqa["frame"]),
                      "e_aurc": best_fiqa["e_aurc"], "aurc": best_fiqa["aurc"],
                      "base_risk": best_fiqa["base_risk"]} if best_fiqa else None),
        },
        "operating_points": {"finqa": finqa_ops, "fiqa": fiqa_ops},
        "validation_metrics": {
            "finqa_exec_verified": FRAMES["finqa_exec_verified"]["base_risk"],
            "finqa_greedy_print_risk": FRAMES["finqa_greedy_print"]["base_risk"],
            "finqa_greedy_exec_risk": FRAMES["finqa_greedy_exec"]["base_risk"],
        },
        "data": {
            "phase2_1_registry": str(REGISTRY_PATH),
            "registry_sha256": sha256_file(REGISTRY_PATH),
            "phase3_freeze_sha256": sha256_file(P3_ROOT / "manifests/phase3_freeze.json"),
        },
        "protected_tests_opened_at_freeze_time": False,
        "important": [
            "Phase-4 did not change the frozen backbone, adapter, prompt policy or selection rule.",
            "Calibration and thresholds were fitted on validation only (5-fold cross-fitting).",
            "Protected test sets were first opened only after this file said FROZEN.",
            "No hyper-parameter (n, temperature, calibrator, coverage target) was chosen on test data.",
        ],
    }
    write_json(PHASE4_FREEZE_PATH, freeze)
    print(json.dumps({k: freeze[k] for k in ["phase", "status", "method_name",
                                             "evg", "calibration", "operating_points"]},
                     indent=2, default=str))
    print("\nPHASE-4 FREEZE WRITTEN:", PHASE4_FREEZE_PATH)
else:
    freeze = read_json(PHASE4_FREEZE_PATH) if PHASE4_FREEZE_PATH.exists() else None
    print("P4-E7 skipped; freeze on disk:", bool(freeze),
          "| status:", (freeze or {}).get("status"))

{
  "phase": "Phase-4",
  "status": "FROZEN",
  "method_name": "EVG-CAS",
  "evg": {
    "n": 8,
    "temperature": 0.7,
    "top_p": 0.9,
    "selection_rule": "exec_verified",
    "tie_break": "mean_token_logprob",
    "uses_gold_answer": false,
    "uses_gold_program": false,
    "uses_only": "the input table, which is part of the model's own prompt"
  },
  "calibration": {
    "split": "validation",
    "folds": 5,
    "selection_rule": "lowest out-of-fold e_AURC on validation among raw/platt/isotonic",
    "finqa": {
      "frame": "finqa_exec_verified",
      "method": "platt_conf_full",
      "selection_rule": "exec_verified",
      "n_candidates": 8,
      "e_aurc": 0.1231987970236064,
      "aurc": 0.1830901402250651,
      "base_risk": 0.3250283125707814
    },
    "fiqa": {
      "frame": "fiqa_greedy",
      "method": "raw_confidence",
      "aggregator": "greedy",
      "e_aurc": 0.2371503180101372,
      "aurc": 0.3620645811493819,
      "base_risk": 0.4529914529914529
  

## 14. Research figures (validation only)

In [18]:
def _savefig(name):
    PLOT_DIR.mkdir(parents=True, exist_ok=True)
    plt.tight_layout()
    plt.savefig(PLOT_DIR / name, dpi=160)
    plt.close()
    print("wrote", PLOT_DIR / name)

# Fig 1 — EVG selection ablation (C1)
if len(P4E3):
    fig, ax = plt.subplots(figsize=(9, 4.6))
    sub = P4E3.sort_values(["rule", "n_candidates"])
    for rule, g in sub.groupby("rule"):
        g = g.sort_values("n_candidates")
        ax.plot(g["n_candidates"], g["answer_accuracy"], marker="o", ls="-",
                label=f"{rule} (answer)")
        ax.plot(g["n_candidates"], g["execution_accuracy"], marker="s", ls="--", alpha=0.5,
                label=f"{rule} (execution)")
    if (METRIC_DIR / "P4-E1.json").exists():
        b = read_json(METRIC_DIR / "P4-E1.json")["FinQA"]
        ax.axhline(b["printed_answer_accuracy"], color="k", lw=1, ls=":",
                   label="frozen greedy printed answer")
        ax.axhline(b["execution_accuracy"], color="k", lw=1, ls="-.",
                   label="frozen greedy executor")
    ax.set_xlabel("number of sampled candidates (n)")
    ax.set_ylabel("accuracy")
    ax.set_title("P4-E3 — selection rule ablation on FinQA validation")
    ax.legend(fontsize=7, ncol=2)
    ax.grid(alpha=0.3)
    _savefig("p4_evg_selection_ablation.png")

# Fig 2 — risk-coverage (C2)
if PHASE4_ROOT.joinpath("manifests/p4_risk_coverage_curves.json").exists():
    curves = read_json(MANIFEST_DIR / "p4_risk_coverage_curves.json")
    fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))
    for ax, task, keys in [(axes[0], "FinQA",
                            ["finqa_greedy_print", "finqa_greedy_exec", "finqa_exec_verified",
                             "finqa_self_consistency"]),
                           (axes[1], "FiQA", ["fiqa_greedy", "fiqa_median_n"])]:
        for k in keys:
            if k not in curves:
                continue
            for style, series in [("raw", curves[k]["raw"]), ("calibrated", curves[k]["calibrated"])]:
                if not series:
                    continue
                ax.plot([p["coverage"] for p in series], [p["risk"] for p in series],
                        marker=("." if style == "raw" else ""),
                        ls=("-" if style == "calibrated" else ":"),
                        label=f"{k} · {style}")
        ax.set_xlabel("coverage")
        ax.set_ylabel("selective risk")
        ax.set_title(f"{task} — risk–coverage (validation, OOF)")
        ax.grid(alpha=0.3)
        ax.legend(fontsize=6)
    _savefig("p4_risk_coverage.png")

# Fig 3 — FiQA sampling aggregation (C3-adjacent)
if len(P4E3b):
    fig, ax = plt.subplots(figsize=(7, 4))
    d = P4E3b.sort_values("mae")
    ax.barh(d["aggregator"], d["mae"], color=["#2a9d8f" if a != "greedy" else "#e76f51" for a in d["aggregator"]])
    ax.set_xlabel("MAE (validation)")
    ax.set_title("P4-E3b — FiQA aggregation over n sampled scores")
    for i, (m, v) in enumerate(zip(d["aggregator"], d["mae"])):
        ax.text(v, i, f" {v:.4f}", va="center", fontsize=8)
    _savefig("p4_fiqa_sampling.png")

print("figures written to", PLOT_DIR)

wrote /home/heet18/Downloads/Research Papers/trust_fin_extra_phase/artifacts/phase4/plots/p4_evg_selection_ablation.png


wrote /home/heet18/Downloads/Research Papers/trust_fin_extra_phase/artifacts/phase4/plots/p4_risk_coverage.png
wrote /home/heet18/Downloads/Research Papers/trust_fin_extra_phase/artifacts/phase4/plots/p4_fiqa_sampling.png
figures written to /home/heet18/Downloads/Research Papers/trust_fin_extra_phase/artifacts/phase4/plots


## 15. Phase-4 completion gates (validation stages)

In [19]:
EXPECTED_OUTPUTS = {
    "P4-E1": METRIC_DIR / "P4-E1.json",
    "P4-E2": PRED_DIR / "P4-E2_candidates.csv",
    "P4-E3": ANALYSIS_DIR / "P4-E3_selection_ablation.csv",
    "P4-E3b": ANALYSIS_DIR / "P4-E3b_fiqa_sampling.csv",
    "P4-E4": ANALYSIS_DIR / "P4-E4_calibration.csv",
    "P4-E5": ANALYSIS_DIR / "P4-E5_calibration.csv",
    "P4-E6": ANALYSIS_DIR / "P4-E6_calibration.csv",
    "P4-E7": MANIFEST_DIR / "phase4_freeze.json",
}
_present = {k: (v.exists()) for k, v in EXPECTED_OUTPUTS.items()}
_required = {k: v for k, v in _present.items() if k in RUN_STAGES or k in {"P4-E1", "P4-E7"}}
print("Output inventory:")
for k, v in EXPECTED_OUTPUTS.items():
    print(f"  {k:8s} {'OK ' if v.exists() else 'MISS'}  {v}")

_missing = [k for k, ok in _present.items() if not ok and k in RUN_STAGES]
freeze_ok = EXPECTED_OUTPUTS["P4-E7"].exists() and read_json(EXPECTED_OUTPUTS["P4-E7"])["status"] == "FROZEN"

print()
print("=" * 80)
print("PHASE-4 STATUS:", "PASS" if not _missing and freeze_ok else "INCOMPLETE")
print("RUN_MODE:", RUN_MODE)
print("Validation stages present:", all(_present[k] for k in EXPECTED_OUTPUTS))
print("Missing (requested):", _missing)
print("Freeze status:", read_json(EXPECTED_OUTPUTS["P4-E7"])["status"] if EXPECTED_OUTPUTS["P4-E7"].exists() else "ABSENT")
print("Protected tests opened so far:", _PROTECTED_STATE["opened"])
print("=" * 80)

Output inventory:
  P4-E1    OK   /home/heet18/Downloads/Research Papers/trust_fin_extra_phase/artifacts/phase4/metrics/P4-E1.json
  P4-E2    OK   /home/heet18/Downloads/Research Papers/trust_fin_extra_phase/artifacts/phase4/predictions/P4-E2_candidates.csv
  P4-E3    OK   /home/heet18/Downloads/Research Papers/trust_fin_extra_phase/artifacts/phase4/analysis/P4-E3_selection_ablation.csv
  P4-E3b   OK   /home/heet18/Downloads/Research Papers/trust_fin_extra_phase/artifacts/phase4/analysis/P4-E3b_fiqa_sampling.csv
  P4-E4    OK   /home/heet18/Downloads/Research Papers/trust_fin_extra_phase/artifacts/phase4/analysis/P4-E4_calibration.csv
  P4-E5    OK   /home/heet18/Downloads/Research Papers/trust_fin_extra_phase/artifacts/phase4/analysis/P4-E5_calibration.csv
  P4-E6    OK   /home/heet18/Downloads/Research Papers/trust_fin_extra_phase/artifacts/phase4/analysis/P4-E6_calibration.csv
  P4-E7    OK   /home/heet18/Downloads/Research Papers/trust_fin_extra_phase/artifacts/phase4/manifests/pha

## 16. P4-E8 — protected test evaluation (dormant until the freeze exists)

This section is **skipped unless** `phase4_freeze.json` says `FROZEN` *and*
`TRUSTFIN4_PROTECTED=1`. Nothing above it may be changed after this cell is first run — that
is the whole point of the freeze.

In [20]:
def require_freeze():
    assert PHASE4_FREEZE_PATH.exists(), "phase4_freeze.json missing — Phase-4 is not frozen."
    fz = read_json(PHASE4_FREEZE_PATH)
    assert fz["status"] == "FROZEN", f"freeze status is {fz['status']!r}"
    assert PROTECTED_OPT_IN, "TRUSTFIN4_PROTECTED must be 1 to open protected tests"
    return fz

PROTECTED_READY = ("P4-E8" in RUN_STAGES) and PHASE4_FREEZE_PATH.exists() \
    and read_json(PHASE4_FREEZE_PATH).get("status") == "FROZEN" and PROTECTED_OPT_IN
print("P4-E8 enabled:", PROTECTED_READY,
      "| freeze:", read_json(PHASE4_FREEZE_PATH).get("status") if PHASE4_FREEZE_PATH.exists() else None,
      "| opt-in:", PROTECTED_OPT_IN)

@torch.inference_mode()
def generate_greedy(model, tokenizer, record, variant: str) -> Dict[str, Any]:
    messages = chat_messages(record, variant=variant)
    prompt_text = render_chat(tokenizer, messages, add_generation_prompt=True)
    enc = tokenizer(prompt_text, return_tensors="pt", truncation=True, max_length=MAX_SEQ_LENGTH)
    device = next(model.parameters()).device
    enc = {k: v.to(device) for k, v in enc.items()}
    t0 = time.perf_counter()
    out = model.generate(**enc, max_new_tokens=max_new_tokens_for(record["dataset"]),
                         do_sample=False, return_dict_in_generate=True, output_scores=True,
                         pad_token_id=tokenizer.pad_token_id, eos_token_id=tokenizer.eos_token_id)
    elapsed = time.perf_counter() - t0
    prompt_len = enc["input_ids"].shape[1]
    gen = out.sequences[:, prompt_len:]
    text = tokenizer.decode(gen[0], skip_special_tokens=True)
    n_gen = int(gen.shape[1])
    mean_logprob, confidence = float("nan"), float("nan")
    if n_gen and getattr(out, "scores", None):
        try:
            trans = model.compute_transition_scores(out.sequences, out.scores, normalize_logits=True)
            vals = trans[0, -n_gen:].detach().float().cpu().numpy()
            vals = vals[np.isfinite(vals)]
            if len(vals):
                mean_logprob = float(np.mean(vals))
                confidence = float(np.exp(np.clip(mean_logprob, -50, 0)))
        except Exception:
            pass
    return {"raw_output": clean_generation(text), "prompt_tokens": int(prompt_len),
            "generated_tokens": n_gen, "latency_seconds": float(elapsed),
            "mean_logprob": mean_logprob, "confidence": confidence}

def greedy_pass(exp_id, records, split, variant_fn) -> pd.DataFrame:
    """Deterministic generation with per-record resume."""
    final_path = PRED_DIR / f"{exp_id}_greedy.csv"
    partial_path = PRED_DIR / f"{exp_id}_greedy.partial.jsonl"
    if final_path.exists() and not OVERWRITE:
        print(f"{exp_id}: using cached greedy generations.")
        return pd.read_csv(final_path)
    done = set()
    if partial_path.exists():
        done = {json.loads(l)["record_id"] for l in partial_path.open() if l.strip()}
        print(f"{exp_id}: resuming, {len(done)} done")
    todo = [r for r in records if r["record_id"] not in done]
    model, tokenizer = get_model()
    t0 = time.perf_counter()
    with partial_path.open("a", encoding="utf-8") as fh:
        for i, rec in enumerate(todo, 1):
            g = generate_greedy(model, tokenizer, rec, variant_fn(rec))
            fh.write(json.dumps({"experiment": exp_id, "record_id": rec["record_id"],
                                 "source_id": rec["source_id"], "dataset": rec["dataset"],
                                 "split": split, **g}, ensure_ascii=False, default=str) + "\n")
            fh.flush()
            if i % 25 == 0 or i == len(todo):
                el = time.perf_counter() - t0
                print(f"{exp_id}: {i}/{len(todo)} | {el/60:.1f} min | {el/max(i,1):.2f} s/rec", flush=True)
    rows = [json.loads(l) for l in partial_path.open() if l.strip()]
    df = pd.DataFrame(rows).drop_duplicates(subset=["record_id"], keep="last")
    df.to_csv(final_path, index=False)
    partial_path.unlink(missing_ok=True)
    return df

def default_variant(rec):
    return PROMPT_VARIANT.get(rec["dataset"], "canonical")

P4-E8 enabled: False | freeze: FROZEN | opt-in: False


In [21]:
# ---------------- P4-E8a — FinQA protected test ----------------
P4E8A = {}
if PROTECTED_READY and "P4-E8" in RUN_STAGES:
    fz = require_freeze()
    finqa_test = read_jsonl(resolve_protected_artifact("finqa_test"))
    print("FinQA test records:", len(finqa_test), Counter(r["dataset"] for r in finqa_test))

    greedy_path = METRIC_DIR / "P4-E8a_greedy.json"
    if greedy_path.exists() and not OVERWRITE:
        P4E8A["greedy"] = read_json(greedy_path)
    else:
        g = greedy_pass("P4-E8a", finqa_test, "test", default_variant)
        rec_by_id = {r["record_id"]: r for r in finqa_test}
        rows = []
        for _, r in g.iterrows():
            rec = rec_by_id.get(r["record_id"])
            if rec is None:
                continue
            parsed = parse_finqa_output(r["raw_output"])
            executable, executed = execute_finqa_program(parsed["program"], str(rec["source_id"]), "test")
            rows.append({
                "record_id": r["record_id"], "raw_output": r["raw_output"],
                "program": parsed["program"], "parsed_answer": parsed["answer"],
                "parse_success": parsed["parse_success"], "confidence": r["confidence"],
                "mean_logprob": r["mean_logprob"],
                "greedy_print_correct": bool(loose_equal(parsed["answer"], rec["target_answer"])),
                "greedy_exec_correct": bool(executable and loose_equal(executed, rec["target_answer"])),
                "program_correct": normalize_program(parsed["program"]) == normalize_program(rec["gold_program"]),
                "program_executable": bool(executable), "executed_answer": executed,
            })
        gd = pd.DataFrame(rows)
        gd.to_csv(PRED_DIR / "P4-E8a_greedy_scored.csv", index=False)
        P4E8A["greedy"] = {
            "n": int(len(gd)),
            "printed_answer_accuracy": float(gd["greedy_print_correct"].mean()),
            "execution_accuracy": float(gd["greedy_exec_correct"].mean()),
            "program_accuracy": float(gd["program_correct"].mean()),
            "parse_success": float(gd["parse_success"].mean()),
            "answer_program_consistency": float(loose_equal_series(gd["parsed_answer"], gd["executed_answer"]).mean()),
        }
        write_json(greedy_path, P4E8A["greedy"])
    print("FinQA TEST (frozen greedy):", json.dumps(P4E8A["greedy"], indent=2))
else:
    print("P4-E8a skipped (not frozen / not opted in).")

P4-E8a skipped (not frozen / not opted in).


In [22]:
# ---------------- P4-E8b — FinQA protected test with EVG ----------------
P4E8B = {}
if PROTECTED_READY and "P4-E8" in RUN_STAGES:
    path = ANALYSIS_DIR / "P4-E8b_evg_selection.csv"
    if path.exists() and not OVERWRITE:
        P4E8B["ablation"] = pd.read_csv(path).to_dict("records")
    else:
        finqa_test = read_jsonl(resolve_protected_artifact("finqa_test"))
        cands = sample_split("P4-E8b", finqa_test, "test", EVG_N, EVG_TEMPERATURE, EVG_TOP_P)
        detail = score_finqa_selection(finqa_test, cands, "P4-E8b", [EVG_N], split="test")
        summary = summarise_finqa(detail)
        detail.to_csv(PRED_DIR / "P4-E8b.csv", index=False)
        summary.to_csv(path, index=False)
        write_json(METRIC_DIR / "P4-E8b.json", summary.to_dict("records"))
        P4E8B["ablation"] = summary.to_dict("records")
        print(summary.to_string(index=False))
    release_model()
else:
    print("P4-E8b skipped (not frozen / not opted in).")

P4-E8b skipped (not frozen / not opted in).


In [23]:
# ---------------- P4-E8c — FiQA protected test (greedy + median-of-n) ----------------
P4E8C = {}
if PROTECTED_READY and "P4-E8" in RUN_STAGES:
    fiqa_test = read_jsonl(resolve_protected_artifact("fiqa_test"))
    out_path = ANALYSIS_DIR / "P4-E8c_fiqa_test.csv"
    if out_path.exists() and not OVERWRITE:
        P4E8C["rows"] = pd.read_csv(out_path).to_dict("records")
    else:
        g = greedy_pass("P4-E8c", fiqa_test, "test", default_variant)
        gidx = {r["record_id"]: r for _, r in g.iterrows()}
        cands = sample_split("P4-E8c", fiqa_test, "test", EVG_N, EVG_TEMPERATURE, EVG_TOP_P)
        cgroup = {str(rid): sub.sort_values("candidate_index").to_dict("records")
                  for rid, sub in cands.groupby("record_id")}
        rows = []
        for rec in fiqa_test:
            gr = gidx.get(rec["record_id"])
            if gr is None:
                continue
            gold = float(rec["target_answer"])
            g_parsed = parse_fiqa_output(str(gr["raw_output"]))
            greedy_pred = (float(np.clip(g_parsed["score"], -1, 1))
                           if g_parsed["score"] is not None else float("nan"))
            cs = cgroup.get(str(rec["record_id"]), [])
            scores = [float(c["score"]) for c in cs if has_value(c.get("score"))]
            logs = [float(c["mean_logprob"]) for c in cs if has_value(c.get("mean_logprob"))]
            # features of the frozen fiqa_median_n frame: confidence, sample_std, mean_logprob
            median_conf = (float(np.mean([np.exp(np.clip(m, -50, 0)) for m in logs]))
                           if logs else float(gr["confidence"]))
            sample_std = float(np.std(scores)) if len(scores) > 1 else 0.0
            mean_logprob = float(np.mean(logs)) if logs else float(gr["mean_logprob"])
            median_pred = float(np.clip(np.median(scores), -1, 1)) if scores else greedy_pred
            rows.append({
                "record_id": rec["record_id"], "gold_score": gold,
                "greedy_pred": greedy_pred, "median_pred": median_pred,
                "greedy_abs_err": abs(greedy_pred - gold) if np.isfinite(greedy_pred) else float("nan"),
                "median_abs_err": abs(median_pred - gold) if np.isfinite(median_pred) else float("nan"),
                "confidence": gr["confidence"], "mean_logprob": gr["mean_logprob"],
                "median_confidence": median_conf, "sample_std": sample_std,
                "median_mean_logprob": mean_logprob,
                "median_correct": bool(np.isfinite(median_pred) and abs(median_pred - gold) <= 0.10),
                "greedy_correct": bool(np.isfinite(greedy_pred) and abs(greedy_pred - gold) <= 0.10),
            })
        d = pd.DataFrame(rows)
        d.to_csv(out_path, index=False)
        def _reg(col_pred, col_err):
            sub = d[d[col_pred].notna()]
            return {"n": int(len(sub)), "mae": float(sub[col_err].mean()),
                    "r2": float(r2_score(sub["gold_score"], sub[col_pred])),
                    "pearson_r": float(pearsonr(sub["gold_score"], sub[col_pred]).statistic),
                    "acc_tol_0.10": float((sub[col_err] <= 0.10).mean())}
        P4E8C = {"greedy": _reg("greedy_pred", "greedy_abs_err"),
                 "median_of_n": _reg("median_pred", "median_abs_err")}
        write_json(METRIC_DIR / "P4-E8c.json", P4E8C)
        print(json.dumps(P4E8C, indent=2))
    release_model()
else:
    print("P4-E8c skipped (not frozen / not opted in).")

P4-E8c skipped (not frozen / not opted in).


In [24]:
# ---------------- P4-E8d — FinBen-FOMC zero-shot OOD (adapter vs base) ----------------
P4E8D = {}
FOMC_VARIANT = "canonical"          # pre-declared: FOMC has no task token in the frozen policy

def parse_fomc_strict(text: str) -> str:
    lab = parse_fomc_output(text)["label"]
    return lab if lab else "unparsed"

if PROTECTED_READY and "P4-E8" in RUN_STAGES:
    fomc = read_jsonl(resolve_protected_artifact("finben_fomc_test"))
    out_path = ANALYSIS_DIR / "P4-E8d_fomc.csv"
    if out_path.exists() and not OVERWRITE:
        dfo = pd.read_csv(out_path)
    else:
        rows = []
        for use_adapter, tag in [(True, "e5_adapter"), (False, "base_model")]:
            release_model()
            clear_memory()
            tokenizer = load_tokenizer(BASE_MODEL_ID)
            model = load_base_model(BASE_MODEL_ID)
            if use_adapter:
                model = PeftModel.from_pretrained(model, str(BACKBONE_ADAPTER))
            model.eval()
            t0 = time.perf_counter()
            for i, rec in enumerate(fomc, 1):
                with torch.inference_mode():
                    g = generate_greedy(model, tokenizer, rec, FOMC_VARIANT)
                gold = str(rec["target_label"]).strip().lower()
                pred = parse_fomc_strict(g["raw_output"])
                rows.append({"system": tag, "record_id": rec["record_id"],
                             "pred": pred, "gold": gold,
                             "correct": pred == gold,
                             "parse_success": parse_fomc_output(g["raw_output"])["parse_success"],
                             "confidence": g["confidence"], "raw_output": g["raw_output"]})
                if i % 50 == 0:
                    print(f"FOMC/{tag}: {i}/{len(fomc)} | {(time.perf_counter()-t0)/60:.1f} min")
            del model, tokenizer
            release_model()
        dfo = pd.DataFrame(rows)
        dfo.to_csv(out_path, index=False)

    for tag, sub in dfo.groupby("system"):
        y_true, y_pred = sub["gold"].tolist(), sub["pred"].tolist()
        labels = sorted(set(y_true) | set(y_pred))
        P4E8D[tag] = {
            "n": int(len(sub)), "accuracy": float((sub["pred"] == sub["gold"]).mean()),
            "macro_f1": float(f1_score(y_true, y_pred, labels=labels, average="macro", zero_division=0)),
            "per_class_f1": {lab: float(f1_score(
                [1 if t == lab else 0 for t in y_true],
                [1 if p == lab else 0 for p in y_pred], zero_division=0)) for lab in labels},
            "parse_success": float(sub["parse_success"].mean()),
        }
    write_json(METRIC_DIR / "P4-E8d.json", P4E8D)
    print(json.dumps(P4E8D, indent=2))
else:
    print("P4-E8d skipped (not frozen / not opted in).")

P4-E8d skipped (not frozen / not opted in).


## 17. Apply the frozen abstention policy to the protected test (P4-E8e)

P4-E8a–E8d measure *what the model answers*. This stage measures *what the frozen policy
would refuse to answer*, which is the only way C2/C3 can be tested: the calibrator and the
operating-point thresholds come from `phase4_freeze.json` and from the calibrator pickles
fitted on validation — **nothing here is refitted, retuned or re-thresholded**.

In [25]:
# ---------------- P4-E8e — frozen abstention applied to the protected test ----------------
P4E8E = {}
if PROTECTED_READY and "P4-E8" in RUN_STAGES:
    import pickle
    fz = read_json(PHASE4_FREEZE_PATH)
    ops = fz.get("operating_points", {})
    assert fz.get("status") == "FROZEN"

    def as_bool(v):
        if isinstance(v, (bool, np.bool_)):
            return bool(v)
        return str(v).strip().lower() in {"true", "1", "yes"}

    def load_frozen_calibrator(meta):
        """Deploy-time calibrator: the object fitted on ALL of validation, as frozen."""
        method = str(meta.get("method", "raw_confidence"))
        frame = meta["frame"]
        if method == "raw_confidence":
            return {"features": [0], "method": method, "path": None,
                    "predict": lambda X: np.clip(np.asarray(X[:, 0], float), 1e-6, 1 - 1e-6)}
        cal_name = "isotonic" if method.startswith("isotonic") else "platt"
        p = MANIFEST_DIR / f"calibrator_{frame}_{cal_name}_final.pkl"
        if not p.exists():
            p = MANIFEST_DIR / f"calibrator_{frame}_{cal_name}.pkl"
        assert p.exists(), f"P4-E8e: frozen calibrator missing for {frame}/{method}: {p}"
        obj = pickle.load(p.open("rb"))
        cols, cal = obj["features"], obj["calibrator"]
        return {"features": cols, "method": method, "path": str(p),
                "predict": lambda X: np.asarray(cal.predict(np.asarray(X, float)[:, cols]), float)}

    def evaluate_abstention(scores, correct, op_points):
        """Apply the *frozen* validation thresholds; no threshold is recomputed here."""
        scores = np.asarray(scores, float)
        correct = np.asarray(correct, bool)
        ok = np.isfinite(scores)
        scores, correct = scores[ok], correct[ok]
        errs = (~correct).astype(float)
        out = {"n": int(len(scores)), "full_accuracy": float(correct.mean()),
               "full_risk": float(errs.mean())}
        for key, op in (op_points or {}).items():
            thr = float(op["threshold"])
            sel = scores >= thr
            out[key] = {
                "threshold": thr,
                "validation_coverage": float(op.get("achieved_coverage", float("nan"))),
                "test_coverage": float(sel.mean()) if len(scores) else float("nan"),
                "validation_selective_risk": float(op.get("validation_selective_risk", float("nan"))),
                "test_selective_risk": float(errs[sel].mean()) if sel.any() else float("nan"),
                "test_selective_accuracy": float(correct[sel].mean()) if sel.any() else float("nan"),
                "answers_abstained": int((~sel).sum()),
                "error_reduction_vs_full": (float(1 - errs[sel].mean() / errs.mean())
                                            if sel.any() and errs.mean() > 0 else float("nan")),
            }
        return out

    # ---- FinQA: rebuild the frozen frame on test from P4-E8b candidates ----
    fin_path = PRED_DIR / "P4-E8b.csv"
    fin_meta = (fz.get("calibration") or {}).get("finqa")
    if fin_path.exists() and fin_meta:
        cal = load_frozen_calibrator(fin_meta)
        d = pd.read_csv(fin_path)
        rule = fin_meta["selection_rule"]
        n_use = int(fin_meta.get("n_candidates", EVG_N))
        sub = d[(d["rule"] == rule) & (d["n_candidates"] == n_use)].copy()
        assert len(sub), f"P4-E8e: no rows for rule={rule} n={n_use} in {fin_path}"
        X = sub[["selected_confidence", "agreement", "executable_rate"]].astype(float).values
        assert max(cal["features"]) < X.shape[1]
        scores = cal["predict"](X)
        sub["calibrated_confidence"] = scores
        sub.to_csv(ANALYSIS_DIR / "P4-E8e_finqa_abstention_scores.csv", index=False)
        res = evaluate_abstention(scores, sub["final_answer_correct"].map(as_bool).values,
                                  ops.get("finqa", {}))
        res.update({"task": "FinQA", "frame": fin_meta["frame"], "method": fin_meta["method"],
                    "selection_rule": rule, "n_candidates": n_use,
                    "calibrator": cal["path"], "features": cal["features"],
                    "base_accuracy": float(sub["final_answer_correct"].map(as_bool).mean())})
        P4E8E["finqa"] = res
        print("FinQA frozen abstention:", json.dumps(res, indent=2, default=str))
    else:
        P4E8E["finqa"] = None
        print("P4-E8e/FinQA: skipped — P4-E8b.csv or frozen FinQA entry missing.")

    # ---- FiQA: rebuild the frozen frame on test from P4-E8c rows ----
    fp = ANALYSIS_DIR / "P4-E8c_fiqa_test.csv"
    fi_meta = (fz.get("calibration") or {}).get("fiqa")
    if fp.exists() and fi_meta:
        cal = load_frozen_calibrator(fi_meta)
        d = pd.read_csv(fp)
        if str(fi_meta.get("aggregator")) == "median_n":
            X = d[["median_confidence", "sample_std", "median_mean_logprob"]].astype(float).values
            ycol = "median_correct"
        else:
            X = d[["confidence", "mean_logprob"]].astype(float).values
            ycol = "greedy_correct"
        assert max(cal["features"]) < X.shape[1]
        scores = cal["predict"](X)
        d["calibrated_confidence"] = scores
        d.to_csv(ANALYSIS_DIR / "P4-E8e_fiqa_abstention_scores.csv", index=False)
        res = evaluate_abstention(scores, d[ycol].map(as_bool).values, ops.get("fiqa", {}))
        res.update({"task": "FiQA", "frame": fi_meta["frame"], "method": fi_meta["method"],
                    "aggregator": fi_meta.get("aggregator"), "correctness_column": ycol,
                    "calibrator": cal["path"], "features": cal["features"],
                    "base_accuracy": float(d[ycol].map(as_bool).mean())})
        P4E8E["fiqa"] = res
        print("FiQA frozen abstention:", json.dumps(res, indent=2, default=str))
    else:
        P4E8E["fiqa"] = None
        print("P4-E8e/FiQA: skipped — P4-E8c_fiqa_test.csv or frozen FiQA entry missing.")

    write_json(METRIC_DIR / "P4-E8e.json", P4E8E)
else:
    print("P4-E8e skipped (not frozen / not opted in).")

P4-E8e skipped (not frozen / not opted in).


## 18. Completion report

Written once the protected stage has run. `phase4_freeze.json` is **never** rewritten here —
its timestamp proves the method was frozen before any protected record was read.

In [26]:
if PROTECTED_READY and "P4-E8" in RUN_STAGES:
    report = {
        "phase": "Phase-4",
        "status": "COMPLETE",
        "completed_at_utc": pd.Timestamp.utcnow().isoformat(),
        "freeze_file": str(PHASE4_FREEZE_PATH),
        "freeze_sha256": sha256_file(PHASE4_FREEZE_PATH),
        "freeze_frozen_at": read_json(PHASE4_FREEZE_PATH).get("frozen_at_utc"),
        "protected_artifacts_opened": sorted(_PROTECTED_STATE.get("opened_keys", [])),
        "run_mode": RUN_MODE,
        "results": {
            "P4-E1": read_json(METRIC_DIR / "P4-E1.json") if (METRIC_DIR / "P4-E1.json").exists() else None,
            "P4-E8a_finqa_test_greedy": P4E8A.get("greedy"),
            "P4-E8b_finqa_test_evg": P4E8B.get("ablation"),
            "P4-E8c_fiqa_test": P4E8C or None,
            "P4-E8d_fomc": P4E8D or None,
            "P4-E8e_frozen_abstention": P4E8E or None,
        },
    }
    write_json(MANIFEST_DIR / "phase4_completion_report.json", report)
    print("PHASE-4 STATUS: COMPLETE — report written")
    print("freeze sha256 (unchanged since before test access):", report["freeze_sha256"])
else:
    print("Completion report written only after P4-E8.")

Completion report written only after P4-E8.


## Outputs to share for Phase-4

```
trust_fin_extra_phase/
├── PLAN.md                                            # design + pre-declared claims
├── README.md
├── docs/exp_4(a).md · exp_4(b).md                     # smoke / full deep-dives
├── notebooks/Phase4_TrustFin_Proposed_Method.ipynb    # executed
└── artifacts/phase4/
    ├── manifests/  phase4_freeze.json, phase4_completion_report.json,
    │               environment snapshot, calibrator_*.pkl, p4_risk_coverage_curves.json
    ├── metrics/    P4-E1 … P4-E8*.json
    ├── analysis/   P4-E3 selection ablation, P4-E4/E5/E6 calibration,
    │               P4-E7_all_calibration, P4-E8*_*.csv
    ├── predictions/ per-record outputs (candidates + greedy, validation and test)
    └── plots/      p4_evg_selection_ablation.png, p4_risk_coverage.png, p4_fiqa_sampling.png
```

Read-only inputs consumed from the outer folder (never modified):
`../artifacts/phase2_1/manifests/*`, `../artifacts/phase3/{manifests,predictions}`,
`../artifacts/phase3/checkpoints/E5`, `../data/raw/finqa*`.